In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

df = pd.read_csv(
    "../data/gold/property_listings.csv"
)

print(df.shape)

(2673, 27)


In [2]:
missing_district = df[
    df["district"].isna()
].copy()

print(
    "Total listings:",
    len(df)
)

print(
    "Missing district:",
    len(missing_district)
)

print(
    "District coverage:",
    f"{df['district'].notna().mean() * 100:.1f}%"
)

Total listings: 2673
Missing district: 411
District coverage: 84.6%


In [3]:
missing_district[
    "source"
].value_counts()


source
harbor-property.com     213
camrealtyservice.com    171
aps.com.kh               27
Name: count, dtype: int64

In [4]:
missing_district[
    [
        "listing_id",
        "source",
        "title",
        "project_name",
        "commune",
        "address",
        "location_text",
        "url"
    ]
].head(30)

,listing_id,source,title,project_name,commune,address,location_text,url
1603,harbor-property.com_109770,harbor-property.com,"Condo for Sale, Prateah Lang, $149000_1Bedroom...",NaN,Prateah Lang,NaN,Prateah Lang,https://www.harbor-property.com/house/detail/1...
1608,harbor-property.com_103533,harbor-property.com,"Condo for Sale, Chroy Chongva, $99900_1Bedroom...",NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1609,harbor-property.com_103532,harbor-property.com,"Condo for Sale, Chroy Chongva, $72000_1Bedroom...",NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1611,harbor-property.com_110922,harbor-property.com,"Condo for Sale, Chroy Chongva, $200000_3Bedroo...",NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1637,harbor-property.com_109421,harbor-property.com,"Condo for Sale, Chroy Chongva, $155000_1Bedroo...",NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1639,harbor-property.com_114213,harbor-property.com,"Condo for Sale, Chroy Chongva, $63000_1Bedroom...",NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1640,harbor-property.com_114133,harbor-property.com,"Condo for Sale, Chroy Chongva, $63000_1Bedroom...",NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1641,harbor-property.com_113791,harbor-property.com,"Condo for Sale, Chroy Chongva, $90000_1Bedroom...",NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1642,harbor-property.com_113992,harbor-property.com,"Condo for Sale, Chroy Chongva, $55000_1Bedroom...",NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1644,harbor-property.com_114151,harbor-property.com,"Condo for Sale, Chroy Chongva, $63000_1Bedroom...",NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...


## Create district patterns

In [5]:
import re
import pandas as pd

DISTRICT_PATTERNS = {
    "Boeung Keng Kang": [
        r"\bboeung keng kang\b",
        r"\bboeng keng kang\b",
        r"\bbkk\s*1\b",
        r"\bbkk\s*2\b",
        r"\bbkk\s*3\b",
        r"\bbkk1\b",
        r"\bbkk2\b",
        r"\bbkk3\b",
    ],

    "Chamkarmon": [
        r"\bchamkarmon\b",
        r"\bchamkar mon\b",
        r"\bchamkar morn\b",
    ],

    "Daun Penh": [
        r"\bdaun penh\b",
        r"\bdoun penh\b",
    ],

    "Chbar Ampov": [
        r"\bchbar ampov\b",
        r"\bchbar ampeou\b",
    ],

    "Russey Keo": [
        r"\brussey keo\b",
        r"\brussei keo\b",
    ],

    "Chroy Changvar": [
        r"\bchroy changvar\b",
        r"\bchroy changva\b",
        r"\bchroy chongva\b",
    ],

    "Sen Sok": [
        r"\bsen sok\b",
    ],

    "Prampi Makara": [
        r"\bprampi makara\b",
        r"\b7 makara\b",
        r"\b7makara\b",
    ],

    "Toul Kork": [
        r"\btoul kork\b",
        r"\btuol kork\b",
    ],

    "Meanchey": [
        r"\bmeanchey\b",
        r"\bmean chey\b",
    ],

    "Pur Senchey": [
        r"\bpur senchey\b",
        r"\bpor sen chey\b",
        r"\bpou senchey\b",
        r"\bporsenchey\b",
    ],
}

## Make a matching function

In [6]:
def find_district_matches(text):
    if pd.isna(text):
        return []

    text = str(text).lower()

    matches = []

    for district, patterns in DISTRICT_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, text):
                matches.append(district)
                break

    return sorted(set(matches))

## Use strong evidence first

Create a text field from:

title, 
project_name, 
commune, 
address, 
location_text, 
url

In [7]:
strong_columns = [
    "title",
    "project_name",
    "commune",
    "address",
    "location_text",
    "url"
]

missing_district["_strong_location_text"] = (
    missing_district[strong_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

In [8]:
missing_district["strong_matches"] = (
    missing_district["_strong_location_text"]
    .apply(find_district_matches)
)

In [9]:
def classify_matches(matches):
    if len(matches) == 0:
        return "not_found"

    if len(matches) == 1:
        return "single_match"

    return "ambiguous"

In [10]:
missing_district["strong_match_status"] = (
    missing_district["strong_matches"]
    .apply(classify_matches)
)

In [11]:
missing_district[
    "strong_match_status"
].value_counts()

strong_match_status
not_found       281
single_match    130
Name: count, dtype: int64

Meaning

single_match
→ only one district found
→ strong candidate ✅

ambiguous
→ more than one district mentioned
→ do not fill automatically ⚠️

not_found
→ no district found
→ we'll try description next

## See which districts were found

In [12]:
single_matches = missing_district[
    missing_district[
        "strong_match_status"
    ] == "single_match"
].copy()

In [13]:
single_matches[
    "district_candidate"
] = single_matches[
    "strong_matches"
].str[0]

In [14]:
single_matches[
    "district_candidate"
].value_counts()

district_candidate
Chroy Changvar      104
Boeung Keng Kang     19
Russey Keo            4
Prampi Makara         3
Name: count, dtype: int64

In [15]:
single_matches[
    [
        "listing_id",
        "source",
        "title",
        "district_candidate",
        "project_name",
        "commune",
        "address",
        "location_text",
        "url"
    ]
].head(30)

,listing_id,source,title,district_candidate,project_name,commune,address,location_text,url
1608,harbor-property.com_103533,harbor-property.com,"Condo for Sale, Chroy Chongva, $99900_1Bedroom...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1609,harbor-property.com_103532,harbor-property.com,"Condo for Sale, Chroy Chongva, $72000_1Bedroom...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1611,harbor-property.com_110922,harbor-property.com,"Condo for Sale, Chroy Chongva, $200000_3Bedroo...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1637,harbor-property.com_109421,harbor-property.com,"Condo for Sale, Chroy Chongva, $155000_1Bedroo...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1639,harbor-property.com_114213,harbor-property.com,"Condo for Sale, Chroy Chongva, $63000_1Bedroom...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1640,harbor-property.com_114133,harbor-property.com,"Condo for Sale, Chroy Chongva, $63000_1Bedroom...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1641,harbor-property.com_113791,harbor-property.com,"Condo for Sale, Chroy Chongva, $90000_1Bedroom...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1642,harbor-property.com_113992,harbor-property.com,"Condo for Sale, Chroy Chongva, $55000_1Bedroom...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1644,harbor-property.com_114151,harbor-property.com,"Condo for Sale, Chroy Chongva, $63000_1Bedroom...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...
1647,harbor-property.com_116149,harbor-property.com,"Condo for Sale, Chroy Chongva, $89000_2Bedroom...",Chroy Changvar,NaN,Chroy Chongva,NaN,Chroy Chongva,https://www.harbor-property.com/house/detail/1...


## find the evidence column

In [16]:
strong_columns = [
    "title",
    "project_name",
    "commune",
    "address",
    "location_text",
    "url"
]


def find_district_evidence(row):

    evidence = []

    for col in strong_columns:

        value = row[col]

        if pd.isna(value):
            continue

        matches = find_district_matches(
            str(value)
        )

        for district in matches:
            evidence.append({
                "column": col,
                "district": district,
                "text": str(value)
            })

    return evidence

In [17]:
single_matches[
    "district_evidence"
] = single_matches.apply(
    find_district_evidence,
    axis=1
)

In [18]:
single_matches[
    [
        "listing_id",
        "source",
        "title",
        "district_candidate",
        "district_evidence"
    ]
].head(30)

,listing_id,source,title,district_candidate,district_evidence
1608,harbor-property.com_103533,harbor-property.com,"Condo for Sale, Chroy Chongva, $99900_1Bedroom...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."
1609,harbor-property.com_103532,harbor-property.com,"Condo for Sale, Chroy Chongva, $72000_1Bedroom...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."
1611,harbor-property.com_110922,harbor-property.com,"Condo for Sale, Chroy Chongva, $200000_3Bedroo...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."
1637,harbor-property.com_109421,harbor-property.com,"Condo for Sale, Chroy Chongva, $155000_1Bedroo...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."
1639,harbor-property.com_114213,harbor-property.com,"Condo for Sale, Chroy Chongva, $63000_1Bedroom...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."
1640,harbor-property.com_114133,harbor-property.com,"Condo for Sale, Chroy Chongva, $63000_1Bedroom...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."
1641,harbor-property.com_113791,harbor-property.com,"Condo for Sale, Chroy Chongva, $90000_1Bedroom...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."
1642,harbor-property.com_113992,harbor-property.com,"Condo for Sale, Chroy Chongva, $55000_1Bedroom...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."
1644,harbor-property.com_114151,harbor-property.com,"Condo for Sale, Chroy Chongva, $63000_1Bedroom...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."
1647,harbor-property.com_116149,harbor-property.com,"Condo for Sale, Chroy Chongva, $89000_2Bedroom...",Chroy Changvar,"[{'column': 'title', 'district': 'Chroy Changv..."


In [19]:
pd.crosstab(
    single_matches["source"],
    single_matches["district_candidate"]
)

district_candidate,Boeung Keng Kang,Chroy Changvar,Prampi Makara,Russey Keo
source,,,,
aps.com.kh,0,4,0,0
camrealtyservice.com,19,3,2,0
harbor-property.com,0,97,1,4


## Create a manual recovery queue

In [20]:
district_review = single_matches[
    [
        "listing_id",
        "source",
        "title",
        "district_candidate",
        "district_evidence",
        "commune",
        "address",
        "location_text",
        "url"
    ]
].copy()

In [21]:
district_review[
    "manual_decision"
] = ""

district_review[
    "manual_note"
] = ""

In [22]:
from pathlib import Path

output_dir = Path(
    "../data/silver/location_recovery"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

district_review.to_csv(
    output_dir /
    "district_recovery_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

In [23]:
def get_evidence_columns(evidence):
    if not evidence:
        return []

    return sorted(
        set(
            item["column"]
            for item in evidence
        )
    )


single_matches[
    "evidence_columns"
] = single_matches[
    "district_evidence"
].apply(
    get_evidence_columns
)

In [24]:
single_matches[
    "evidence_columns"
].value_counts()

evidence_columns
[commune, location_text, title]    102
[title, url]                        17
[title]                              9
[url]                                2
Name: count, dtype: int64

In [25]:
from collections import Counter

evidence_counter = Counter()

for evidence in single_matches[
    "district_evidence"
]:
    for item in evidence:
        evidence_counter[
            item["column"]
        ] += 1

evidence_counter

Counter({'title': 128, 'commune': 102, 'location_text': 102, 'url': 19})

## can also test consistency

For each candidate, check whether every strong field that mentions a district agrees with the candidate.

In [26]:
def evidence_is_consistent(evidence):
    districts = {
        item["district"]
        for item in evidence
    }

    return len(districts) == 1


single_matches[
    "evidence_consistent"
] = single_matches[
    "district_evidence"
].apply(
    evidence_is_consistent
)

In [27]:
single_matches[
    "evidence_consistent"
].value_counts()

evidence_consistent
True    130
Name: count, dtype: int64

## After validation: apply the 130 recoveries

In [28]:
location_df = df.copy()

In [29]:
location_df[
    "district_original"
] = location_df["district"]

location_df[
    "district_recovery_source"
] = None

In [30]:
recovery_map = dict(
    zip(
        single_matches["listing_id"],
        single_matches["district_candidate"]
    )
)

In [31]:
mask = (
    location_df["district"].isna()
    &
    location_df["listing_id"].isin(
        recovery_map
    )
)

location_df.loc[
    mask,
    "district"
] = (
    location_df.loc[
        mask,
        "listing_id"
    ]
    .map(recovery_map)
)

location_df.loc[
    mask,
    "district_recovery_source"
] = "strong_text_match"

In [32]:
print("Total records:", len(location_df))

print(
    "Recovered districts:",
    mask.sum()
)

print(
    "Remaining missing:",
    location_df["district"].isna().sum()
)

print(
    "District coverage:",
    f"{location_df['district'].notna().mean() * 100:.1f}%"
)

Total records: 2673
Recovered districts: 130
Remaining missing: 281
District coverage: 89.5%


## Check the recovered district distribution

In [33]:
location_df[
    "district"
].value_counts()

district
Boeung Keng Kang    578
Meanchey            416
Chroy Changvar      305
Chamkarmon          298
Toul Kork           295
Sen Sok             233
Chbar Ampov          92
Daun Penh            69
Prampi Makara        52
Russey Keo           46
Pur Senchey           8
Name: count, dtype: int64

In [34]:
location_df[
    location_df[
        "district_recovery_source"
    ] == "strong_text_match"
][
    "district"
].value_counts()

district
Chroy Changvar      104
Boeung Keng Kang     19
Russey Keo            4
Prampi Makara         3
Name: count, dtype: int64

## Now work on the remaining 281

These are the records where:

title, 
URL, 
commune, 
address, 
location_text, 
project_name

didn't give us a district.

In [35]:
remaining_missing = location_df[
    location_df["district"].isna()
].copy()

print(
    "Remaining missing:",
    len(remaining_missing)
)

Remaining missing: 281


In [36]:
remaining_missing[
    "description_matches"
] = (
    remaining_missing["description"]
    .fillna("")
    .apply(find_district_matches)
)

In [37]:
remaining_missing[
    "description_match_status"
] = (
    remaining_missing[
        "description_matches"
    ]
    .apply(classify_matches)
)

In [38]:
remaining_missing[
    "description_match_status"
].value_counts()

description_match_status
single_match    160
not_found       117
ambiguous         4
Name: count, dtype: int64

Suppose the description says:

Beautiful condo in Sen Sok,
only 10 minutes from Toul Kork.

Our matcher could see:

Sen Sok
Toul Kork

and classify it:

ambiguous

## See what description recovered

In [39]:
description_single = remaining_missing[
    remaining_missing[
        "description_match_status"
    ] == "single_match"
].copy()

In [40]:
description_single[
    "district_candidate"
] = (
    description_single[
        "description_matches"
    ].str[0]
)

In [41]:
print(
    "Description candidates:",
    len(description_single)
)

description_single[
    "district_candidate"
].value_counts()

Description candidates: 160


district_candidate
Boeung Keng Kang    49
Chamkarmon          25
Toul Kork           22
Chroy Changvar      21
Sen Sok             19
Daun Penh           10
Meanchey             8
Prampi Makara        4
Chbar Ampov          2
Name: count, dtype: int64

In [42]:
description_single[
    [
        "listing_id",
        "source",
        "title",
        "description",
        "district_candidate"
    ]
].head(30)

,listing_id,source,title,description,district_candidate
1755,harbor-property.com_115162,harbor-property.com,"Condo for Sale, Phnom Penh",Resell unit at Le Code BKk1 Spacial point of U...,Boeung Keng Kang
1756,harbor-property.com_115159,harbor-property.com,"Condo for Sale, Phnom Penh",Resell unit at Le Code BKk1 Spacial point of U...,Boeung Keng Kang
1783,harbor-property.com_114782,harbor-property.com,"Condo for Sale, Orussey I, $442642_2Bedroom_13...",Condo for sale 公寓出售 Property code: ACD26-027 P...,Prampi Makara
1784,harbor-property.com_114781,harbor-property.com,"Condo for Sale, Orussey I, $240162_1Bedroom_76m²",Condo for sale 公寓出售 Property code: ACD26-026 P...,Prampi Makara
1851,harbor-property.com_114372,harbor-property.com,"Condo for Sale, Orussey II, $140346_48m²",Condo for sale 公寓出售 Property code: ACD26-012 P...,Prampi Makara
1880,harbor-property.com_114267,harbor-property.com,"Condo for Sale, Tuek Thla, $70700_43m²","Studio Condo of 43 Sqm on 17F for US$70,700 at...",Sen Sok
1881,harbor-property.com_114266,harbor-property.com,"Condo for Sale, Tuek Thla, $94700_53m²",Studio with Balcony of 53 Sqm on 20F for US$94...,Sen Sok
1882,harbor-property.com_114264,harbor-property.com,"Condo for Sale, Tuek Thla, $106200_1Bedroom_64m²","One-Bedroom Condo of 64 Sqm on 15F for US$106,...",Sen Sok
1883,harbor-property.com_114263,harbor-property.com,"Condo for Sale, Tuek Thla, $126300_2Bedroom_77m²","Two-Bedroom Condo of 77 Sqm on 10F for US$126,...",Sen Sok
1884,harbor-property.com_114262,harbor-property.com,"Condo for Sale, Tuek Thla, $139400_2Bedroom_87m²",2 Beds 2 Bath Condo of 87 Sqm on 10F for US$13...,Sen Sok


## classify description evidence

We now want to separate descriptions that clearly say the property is located in a district from weaker mentions like “near”, “close to”, or “10 minutes from”.

In [43]:
LOCATION_PHRASES = [
    r"located in\s+{}",
    r"location[:\s]+{}",
    r"situated in\s+{}",
    r"property in\s+{}",
    r"condo in\s+{}",
    r"apartment in\s+{}",
    r"penthouse in\s+{}",
    r"for sale in\s+{}",
]

In [44]:
def strong_description_location(text, district):
    if pd.isna(text):
        return False

    text = str(text).lower()

    patterns = DISTRICT_PATTERNS[district]

    for district_pattern in patterns:
        for template in LOCATION_PHRASES:
            pattern = template.format(
                district_pattern
            )

            if re.search(
                pattern,
                text,
                flags=re.IGNORECASE
            ):
                return True

    return False

In [45]:
description_single[
    "description_strong"
] = description_single.apply(
    lambda row: strong_description_location(
        row["description"],
        row["district_candidate"]
    ),
    axis=1
)

In [46]:
description_single[
    "description_strong"
].value_counts()

description_strong
False    129
True      31
Name: count, dtype: int64

## Also inspect the weak ones

In [47]:
description_single[
    description_single[
        "description_strong"
    ] == False
][
    [
        "listing_id",
        "source",
        "title",
        "district_candidate",
        "description"
    ]
].head(30)

,listing_id,source,title,district_candidate,description
1755,harbor-property.com_115162,harbor-property.com,"Condo for Sale, Phnom Penh",Boeung Keng Kang,Resell unit at Le Code BKk1 Spacial point of U...
1756,harbor-property.com_115159,harbor-property.com,"Condo for Sale, Phnom Penh",Boeung Keng Kang,Resell unit at Le Code BKk1 Spacial point of U...
1783,harbor-property.com_114782,harbor-property.com,"Condo for Sale, Orussey I, $442642_2Bedroom_13...",Prampi Makara,Condo for sale 公寓出售 Property code: ACD26-027 P...
1784,harbor-property.com_114781,harbor-property.com,"Condo for Sale, Orussey I, $240162_1Bedroom_76m²",Prampi Makara,Condo for sale 公寓出售 Property code: ACD26-026 P...
1851,harbor-property.com_114372,harbor-property.com,"Condo for Sale, Orussey II, $140346_48m²",Prampi Makara,Condo for sale 公寓出售 Property code: ACD26-012 P...
1924,harbor-property.com_113889,harbor-property.com,"Condo for Sale, Phsar Thmey I, $160000_2Bedroo...",Daun Penh,Condo for sale 公寓出售 Property code: ACD26-005 S...
1925,harbor-property.com_113888,harbor-property.com,"Condo for Sale, Phsar Thmey I, $120000_1Bedroo...",Daun Penh,Condo for sale 公寓出售 Property code: ACD26-004 S...
1926,harbor-property.com_113877,harbor-property.com,"Condo for Sale, Phsar Thmey I, $90000_40.97m²",Daun Penh,Condo for sale 公寓出售 Property code: ACD26-003 S...
1928,harbor-property.com_113856,harbor-property.com,"Condo for Sale, Phnom Penh, $99000_1Bedroom_39...",Boeung Keng Kang,គំរោង : Le Conde BKK1 Type : Studio Room Size ...
1932,harbor-property.com_113688,harbor-property.com,"Condo for Sale, Tuol Sangkae II, $88000_2Bedro...",Toul Kork,ខុនដូរ ជួល ឬ លក់ (88000$) អត់កាត់ថ្លៃ (016 555...


## Apply the 31 strong description matches

In [48]:
description_strong_df = description_single[
    description_single["description_strong"] == True
].copy()

description_recovery_map = dict(
    zip(
        description_strong_df["listing_id"],
        description_strong_df["district_candidate"]
    )
)

In [49]:
description_mask = (
    location_df["district"].isna()
    &
    location_df["listing_id"].isin(
        description_recovery_map
    )
)

location_df.loc[
    description_mask,
    "district"
] = (
    location_df.loc[
        description_mask,
        "listing_id"
    ]
    .map(description_recovery_map)
)

location_df.loc[
    description_mask,
    "district_recovery_source"
] = "strong_description_match"

In [50]:
print(
    "Description recovered:",
    description_mask.sum()
)

print(
    "Remaining missing:",
    location_df["district"].isna().sum()
)

print(
    "District coverage:",
    f"{location_df['district'].notna().mean() * 100:.1f}%"
)

print(
    "Total rows:",
    len(location_df)
)

Description recovered: 31
Remaining missing: 250
District coverage: 90.6%
Total rows: 2673


## Learn commune → district relationships from known records

Your dataset already has 2,423 district labels after recovery.

We can use those records to see whether a commune consistently belongs to one district.

In [51]:
known_location = location_df[
    location_df["district"].notna()
    &
    location_df["commune"].notna()
].copy()

In [52]:
commune_district_counts = (
    known_location
    .groupby(
        ["commune", "district"]
    )
    .size()
    .reset_index(
        name="count"
    )
)

commune_district_counts

,commune,district,count
0,7 Makara,Prampi Makara,1
1,BKK 1,Boeung Keng Kang,104
2,BKK 2,Boeung Keng Kang,3
3,BKK 3,Boeung Keng Kang,36
4,Bkk1,Boeung Keng Kang,122
...,...,...,...
61,Veal Vong,Prampi Makara,40
62,Wat Phnom,Daun Penh,14
63,chaktomuk,Daun Penh,1
64,olympic,Chamkarmon,3


In [53]:
commune_summary = (
    commune_district_counts
    .sort_values(
        ["commune", "count"],
        ascending=[True, False]
    )
)

commune_summary.head(50)

,commune,district,count
0,7 Makara,Prampi Makara,1
1,BKK 1,Boeung Keng Kang,104
2,BKK 2,Boeung Keng Kang,3
3,BKK 3,Boeung Keng Kang,36
4,Bkk1,Boeung Keng Kang,122
5,Bkk2,Boeung Keng Kang,2
6,Bkk3,Boeung Keng Kang,24
7,Boeng Reang,Daun Penh,5
8,Boeng Tompun I,Meanchey,15
9,Boeng Tompun Ii,Meanchey,4


## Create only high-confidence mappings

In [54]:
commune_stats = (
    known_location
    .groupby("commune")
    .agg(
        total=("listing_id", "count"),
        district_count=(
            "district",
            "nunique"
        )
    )
)

commune_stats.head()

,total,district_count
commune,,
7 Makara,1,1
BKK 1,104,1
BKK 2,3,1
BKK 3,36,1
Bkk1,122,1


Keep only communes that:

appear in at least 3 known listings
AND
always map to exactly one district

In [55]:
safe_communes = commune_stats[
    (commune_stats["total"] >= 3)
    &
    (commune_stats["district_count"] == 1)
].index

In [56]:
commune_to_district = (
    known_location[
        known_location[
            "commune"
        ].isin(safe_communes)
    ]
    .drop_duplicates(
        subset=["commune"]
    )
    .set_index(
        "commune"
    )["district"]
    .to_dict()
)

In [57]:
commune_to_district

{'Boeung Kak 1': 'Toul Kork',
 'Phnom Penh Thmey': 'Sen Sok',
 'BKK 1': 'Boeung Keng Kang',
 'Srah Chak': 'Daun Penh',
 'Chak Angrae Leu': 'Meanchey',
 'Stueng Mean chey': 'Meanchey',
 'Toul Tum Poung 1': 'Chamkarmon',
 "Ou Baek K'am": 'Sen Sok',
 'Wat Phnom': 'Daun Penh',
 'Nirouth': 'Chbar Ampov',
 'Boeung Tumpun': 'Meanchey',
 'BKK 3': 'Boeung Keng Kang',
 'Tuek Thla': 'Sen Sok',
 'Chroy Changvar': 'Chroy Changvar',
 'Tonle Bassac': 'Chamkarmon',
 'Boeung Trabek': 'Chamkarmon',
 'Boeung Tumpun 2': 'Meanchey',
 'Veal Vong': 'Prampi Makara',
 'Kouk Khleang': 'Sen Sok',
 'Boeng Reang': 'Daun Penh',
 'Toul Svay Prey 1': 'Boeung Keng Kang',
 'Russey Keo': 'Russey Keo',
 'Olympic': 'Boeung Keng Kang',
 'Tuol Sangke': 'Russey Keo',
 'Boeung Tumpun 1': 'Meanchey',
 'Boeung Kak 2': 'Toul Kork',
 'Tumnob Tuek': 'Boeung Keng Kang',
 'Tuol Sangkae 2': 'Russey Keo',
 'Preaek Lieb': 'Chroy Changvar',
 'BKK 2': 'Boeung Keng Kang',
 'Phnom Penh Thmei': 'Sen Sok',
 'Boeng Tompun I': 'Meanchey',
 'Bo

## Try those mappings on the 250 remaining records

In [58]:
remaining_location = location_df[
    location_df["district"].isna()
].copy()

print(len(remaining_location))

250


In [59]:
remaining_location[
    "commune_candidate"
] = remaining_location[
    "commune"
].map(
    commune_to_district
)

In [60]:
print(
    "Recoverable from commune:",
    remaining_location[
        "commune_candidate"
    ].notna().sum()
)

remaining_location[
    "commune_candidate"
].value_counts()

Recoverable from commune: 30


commune_candidate
Sen Sok        20
Chbar Ampov     9
Daun Penh       1
Name: count, dtype: int64

## Apply the 30 commune recoveries

In [61]:
commune_recovery_map = dict(
    zip(
        remaining_location["listing_id"],
        remaining_location["commune_candidate"]
    )
)

In [62]:
commune_recovery_map = {
    listing_id: district
    for listing_id, district in commune_recovery_map.items()
    if pd.notna(district)
}

In [63]:
commune_mask = (
    location_df["district"].isna()
    &
    location_df["listing_id"].isin(
        commune_recovery_map
    )
)

location_df.loc[
    commune_mask,
    "district"
] = (
    location_df.loc[
        commune_mask,
        "listing_id"
    ]
    .map(commune_recovery_map)
)

location_df.loc[
    commune_mask,
    "district_recovery_source"
] = "commune_mapping"

In [64]:
print(
    "Recovered from commune:",
    commune_mask.sum()
)

print(
    "Remaining missing:",
    location_df["district"].isna().sum()
)

print(
    "District coverage:",
    f"{location_df['district'].notna().mean() * 100:.1f}%"
)

print(
    "Total rows:",
    len(location_df)
)

Recovered from commune: 30
Remaining missing: 220
District coverage: 91.8%
Total rows: 2673


## Build neighborhood → district mapping

First use listings that now have both a district and location information:

In [65]:
known_df = location_df[
    location_df["district"].notna()
].copy()

In [66]:
commune_mapping_stats = (
    known_df[
        known_df["commune"].notna()
    ]
    .groupby("commune")
    .agg(
        records=("listing_id", "count"),
        district_count=("district", "nunique"),
        district=(
            "district",
            lambda x: x.mode().iloc[0]
        )
    )
    .sort_values(
        "records",
        ascending=False
    )
)

commune_mapping_stats.head(30)

,records,district_count,district
commune,,,
Tonle Bassac,158,1,Chamkarmon
Chak Angrae Leu,157,1,Meanchey
Bkk1,122,1,Boeung Keng Kang
Boeung Kak I,117,1,Toul Kork
BKK 1,104,1,Boeung Keng Kang
Chroy Chongva,97,1,Chroy Changvar
Chroy Changvar,89,1,Chroy Changvar
Boeung Tumpun,70,1,Meanchey
Nirouth,59,1,Chbar Ampov


We only want names where:

records >= 3
AND
district_count == 1

In [67]:
safe_neighborhoods = (
    commune_mapping_stats[
        (commune_mapping_stats["records"] >= 3)
        &
        (commune_mapping_stats["district_count"] == 1)
    ]
)

safe_neighborhoods

,records,district_count,district
commune,,,
Tonle Bassac,158,1,Chamkarmon
Chak Angrae Leu,157,1,Meanchey
Bkk1,122,1,Boeung Keng Kang
Boeung Kak I,117,1,Toul Kork
BKK 1,104,1,Boeung Keng Kang
Chroy Chongva,97,1,Chroy Changvar
Chroy Changvar,89,1,Chroy Changvar
Boeung Tumpun,70,1,Meanchey
Nirouth,59,1,Chbar Ampov


This table is important because it tells us:

When this neighborhood/commune appears in the clean dataset, does it consistently belong to one district?

## Search these neighborhood names inside remaining titles

In [68]:
remaining_missing = location_df[
    location_df["district"].isna()
].copy()

print(
    "Remaining:",
    len(remaining_missing)
)

Remaining: 220


In [69]:
neighborhood_to_district = (
    safe_neighborhoods[
        "district"
    ].to_dict()
)

In [70]:
def find_neighborhood_district(title):

    if pd.isna(title):
        return None

    text = str(title).lower()

    matches = []

    for neighborhood, district in (
        neighborhood_to_district.items()
    ):

        neighborhood_text = (
            str(neighborhood)
            .lower()
            .strip()
        )

        if neighborhood_text in text:
            matches.append(
                (
                    neighborhood,
                    district
                )
            )

    # Nothing found
    if len(matches) == 0:
        return None

    # Find unique districts
    districts = {
        district
        for _, district in matches
    }

    # Only accept when every matched
    # neighborhood agrees
    if len(districts) == 1:
        return list(districts)[0]

    return None

In [71]:
remaining_missing[
    "title_neighborhood_candidate"
] = (
    remaining_missing["title"]
    .apply(
        find_neighborhood_district
    )
)

In [72]:
print(
    "Recoverable from title neighborhood:",
    remaining_missing[
        "title_neighborhood_candidate"
    ].notna().sum()
)

Recoverable from title neighborhood: 1


In [73]:
remaining_missing[
    "title_neighborhood_candidate"
].value_counts()

title_neighborhood_candidate
Sen Sok    1
Name: count, dtype: int64

## inspect that one candidate:

In [74]:
remaining_missing[
    remaining_missing[
        "title_neighborhood_candidate"
    ].notna()
][
    [
        "listing_id",
        "source",
        "title",
        "commune",
        "location_text",
        "title_neighborhood_candidate"
    ]
]

,listing_id,source,title,commune,location_text,title_neighborhood_candidate
2612,aps.com.kh_2-bedroom-condo-for-sale-urban-loft...,aps.com.kh,2 Bedroom Condo For Sale - Urban Loft | Phnom ...,NaN,NaN,Sen Sok


In [75]:
remaining_missing[
    "title_neighborhood_candidate"
].value_counts()

title_neighborhood_candidate
Sen Sok    1
Name: count, dtype: int64

In [76]:
title_recovery = remaining_missing[
    remaining_missing[
        "title_neighborhood_candidate"
    ].notna()
].copy()

title_recovery_map = dict(
    zip(
        title_recovery["listing_id"],
        title_recovery[
            "title_neighborhood_candidate"
        ]
    )
)

title_mask = (
    location_df["district"].isna()
    &
    location_df["listing_id"].isin(
        title_recovery_map
    )
)

location_df.loc[
    title_mask,
    "district"
] = (
    location_df.loc[
        title_mask,
        "listing_id"
    ].map(title_recovery_map)
)

location_df.loc[
    title_mask,
    "district_recovery_source"
] = "title_neighborhood_match"

In [77]:
print("Recovered:", title_mask.sum())
print(
    "Remaining missing:",
    location_df["district"].isna().sum()
)
print(
    "Coverage:",
    f"{location_df['district'].notna().mean() * 100:.1f}%"
)
print("Rows:", len(location_df))

Recovered: 1
Remaining missing: 219
Coverage: 91.8%
Rows: 2673


## Now save the location-enriched dataset:

In [78]:
from pathlib import Path

output_dir = Path(
    "../data/gold"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

location_df.to_csv(
    output_dir /
    "property_listings_location_enriched.csv",
    index=False,
    encoding="utf-8-sig"
)

In [79]:
print(location_df.shape)

(2673, 29)


# Coordinate Recovery 📍

## Check available location information

In [80]:
location_fields = [
    "project_name",
    "address",
    "location_text",
    "commune",
    "district"
]

for col in location_fields:
    available = location_df[col].notna().sum()
    coverage = (
        location_df[col].notna().mean() * 100
    )

    print(
        f"{col:<20}: "
        f"{available:>4} / {len(location_df)} "
        f"({coverage:.1f}%)"
    )

project_name        :  439 / 2673 (16.4%)
address             :  937 / 2673 (35.1%)
location_text       : 2475 / 2673 (92.6%)
commune             : 1600 / 2673 (59.9%)
district            : 2454 / 2673 (91.8%)


This tells us what coordinate-recovery strategy is realistic.

## Create a geocoding quality level

In [81]:
def choose_geocode_level(row):

    if pd.notna(row["project_name"]):
        return "project"

    if pd.notna(row["address"]):
        return "address"

    if pd.notna(row["location_text"]):
        return "location_text"

    if (
        pd.notna(row["commune"])
        and pd.notna(row["district"])
    ):
        return "commune"

    if pd.notna(row["district"]):
        return "district_only"

    return "insufficient"

In [82]:
location_df["geocode_level"] = (
    location_df.apply(
        choose_geocode_level,
        axis=1
    )
)

In [83]:
location_df[
    "geocode_level"
].value_counts()

geocode_level
location_text    1538
address           498
project           439
insufficient      146
district_only      52
Name: count, dtype: int64

## Build geocoding queries

Don't geocode yet. First create the search strings that we would send to a map/geocoding service.

In [84]:
def build_geocode_query(row):

    district = (
        str(row["district"]).strip()
        if pd.notna(row["district"])
        else ""
    )

    commune = (
        str(row["commune"]).strip()
        if pd.notna(row["commune"])
        else ""
    )

    if row["geocode_level"] == "project":

        return (
            f"{row['project_name']}, "
            f"{district}, Phnom Penh, Cambodia"
        )

    elif row["geocode_level"] == "address":

        return (
            f"{row['address']}, "
            f"{district}, Phnom Penh, Cambodia"
        )

    elif row["geocode_level"] == "location_text":

        return (
            f"{row['location_text']}, "
            f"{district}, Phnom Penh, Cambodia"
        )

    elif row["geocode_level"] == "commune":

        return (
            f"{commune}, "
            f"{district}, Phnom Penh, Cambodia"
        )

    elif row["geocode_level"] == "district_only":

        return (
            f"{district}, "
            "Phnom Penh, Cambodia"
        )

    return None

In [85]:
location_df["geocode_query"] = (
    location_df.apply(
        build_geocode_query,
        axis=1
    )
)

In [86]:
location_df[
    [
        "listing_id",
        "project_name",
        "commune",
        "district",
        "geocode_level",
        "geocode_query"
    ]
].head(30)

,listing_id,project_name,commune,district,geocode_level,geocode_query
0,realestate.com.kh_246560,Time Square II,Boeung Kak 1,Toul Kork,project,"Time Square II, Toul Kork, Phnom Penh, Cambodia"
1,realestate.com.kh_259255,Chip Mong | Park Land TK Condo,Phnom Penh Thmey,Sen Sok,project,"Chip Mong | Park Land TK Condo, Sen Sok, Phnom..."
2,realestate.com.kh_248268,Time Square 306,BKK 1,Boeung Keng Kang,project,"Time Square 306, Boeung Keng Kang, Phnom Penh,..."
3,realestate.com.kh_255697,One Park,Srah Chak,Daun Penh,project,"One Park, Daun Penh, Phnom Penh, Cambodia"
4,realestate.com.kh_247259,TK Star International,Boeung Kak 1,Toul Kork,project,"TK Star International, Toul Kork, Phnom Penh, ..."
5,realestate.com.kh_240206,NaN,BKK 1,Boeung Keng Kang,address,"BKK 1, Boeng Keng Kang, Phnom Penh, Boeung Ken..."
6,realestate.com.kh_236155,Urban Village Phase 2,Chak Angrae Leu,Meanchey,project,"Urban Village Phase 2, Meanchey, Phnom Penh, C..."
7,realestate.com.kh_267352,Time Square 306,BKK 1,Boeung Keng Kang,project,"Time Square 306, Boeung Keng Kang, Phnom Penh,..."
8,realestate.com.kh_267340,Urban Village Phase 2,Chak Angrae Leu,Meanchey,project,"Urban Village Phase 2, Meanchey, Phnom Penh, C..."
9,realestate.com.kh_267336,NaN,Boeung Kak 1,Toul Kork,address,", Boeung Kak 1, Toul Kork, Phnom Penh, Toul Ko..."


## Count unique queries

This is important because many listings may belong to the same building.

For example, you do not want to geocode:

Time Square 5, 
Time Square 5, 
Time Square 5, 
Time Square 5

50 separate times.

Instead geocode the building once and reuse the coordinates.

In [87]:
print(
    "Total records:",
    len(location_df)
)

print(
    "Records with query:",
    location_df[
        "geocode_query"
    ].notna().sum()
)

print(
    "Unique geocode queries:",
    location_df[
        "geocode_query"
    ].nunique()
)

Total records: 2673
Records with query: 2527
Unique geocode queries: 214


In [88]:
location_df[
    "geocode_level"
].value_counts()

geocode_level
location_text    1538
address           498
project           439
insufficient      146
district_only      52
Name: count, dtype: int64

## audit the 214 queries

First check how many unique queries come from each precision level:

In [89]:
query_summary = (
    location_df[
        location_df["geocode_query"].notna()
    ]
    .groupby("geocode_level")
    .agg(
        records=("listing_id", "count"),
        unique_queries=("geocode_query", "nunique")
    )
)

query_summary

,records,unique_queries
geocode_level,,
address,498,79
district_only,52,4
location_text,1538,54
project,439,77


## Check the most repeated queries

In [90]:
query_counts = (
    location_df[
        "geocode_query"
    ]
    .value_counts()
)

query_counts.head(30)

geocode_query
Boeung Keng Kang, Boeung Keng Kang, Phnom Penh, Cambodia                            235
Bkk1, Boeung Keng Kang, Phnom Penh, Cambodia                                        122
Boeung Kak I, Toul Kork, Phnom Penh, Cambodia                                       117
Meanchey, Meanchey, Phnom Penh, Cambodia                                            116
Sen Sok, Sen Sok, Phnom Penh, Cambodia                                              107
Chroy Chongva, Chroy Changvar, Phnom Penh, Cambodia                                  97
Chroy Changvar, Chroy Changvar, Phnom Penh, Cambodia                                 83
Toul Kork, Toul Kork, Phnom Penh, Cambodia                                           83
Chamkarmon, Chamkarmon, Phnom Penh, Cambodia                                         83
Chak Angrae Leu, Meanchey, Phnom Penh, Cambodia                                      68
Urban Village Phase 2, Meanchey, Phnom Penh, Cambodia                                59
Tonle Bassac, Cham

## Most important: inspect location_text

In [91]:
location_text_queries = (
    location_df[
        location_df["geocode_level"]
        == "location_text"
    ][
        [
            "listing_id",
            "title",
            "location_text",
            "district",
            "geocode_query"
        ]
    ]
)

location_text_queries.head(30)

,listing_id,title,location_text,district,geocode_query
927,khpropertyhub.com_5517,1 Bedroom Condo at Saen Sok for sale,Sen Sok,Sen Sok,"Sen Sok, Sen Sok, Phnom Penh, Cambodia"
928,khpropertyhub.com_23338,Urban Loft- 1Bedroom for Sale Corner Type,Sen Sok,Sen Sok,"Sen Sok, Sen Sok, Phnom Penh, Cambodia"
929,khpropertyhub.com_23419,2 bedroom urgent sale,Chroy Changvar,Chroy Changvar,"Chroy Changvar, Chroy Changvar, Phnom Penh, Ca..."
930,khpropertyhub.com_23406,Luxury 2 Bedroom Condo for Sale in BKK1,Boeung Keng Kang,Boeung Keng Kang,"Boeung Keng Kang, Boeung Keng Kang, Phnom Penh..."
931,khpropertyhub.com_23371,Luxury 3Bedroom Penthouse for Sale at Time Squ...,Toul Kork,Toul Kork,"Toul Kork, Toul Kork, Phnom Penh, Cambodia"
932,khpropertyhub.com_23365,High Floor 2 Bedroom Apartment For Sale at Tim...,Toul Kork,Toul Kork,"Toul Kork, Toul Kork, Phnom Penh, Cambodia"
933,khpropertyhub.com_23362,Fully Furnished 2 Bedroom Condo for Sale in Ti...,Toul Kork,Toul Kork,"Toul Kork, Toul Kork, Phnom Penh, Cambodia"
934,khpropertyhub.com_23343,2 bedroom under Market Price.,Meanchey,Meanchey,"Meanchey, Meanchey, Phnom Penh, Cambodia"
935,khpropertyhub.com_23326,ខុនដូ Luxury កោះនរាហ៍តម្លៃល្អ,Chbar Ampov,Chbar Ampov,"Chbar Ampov, Chbar Ampov, Phnom Penh, Cambodia"
936,khpropertyhub.com_23260,Urban village phase II,Meanchey,Meanchey,"Meanchey, Meanchey, Phnom Penh, Cambodia"


In [92]:
location_text_queries[
    "location_text"
].value_counts().head(30)

location_text
Boeung Keng Kang      235
Bkk1                  122
Boeung Kak I          117
Meanchey              116
Sen Sok               107
Chroy Chongva          97
Chroy Changvar         83
Toul Kork              83
Chamkarmon             83
Chak Angrae Leu        68
Tonle Bassac           53
Phnom Penh             45
Chbar Ampov            34
Daun Penh              28
Phnom Penh Thmei       28
Tuek Thla              25
Bkk3                   24
Boeung Trobaek         19
Russey Keo             18
Stueng Mean Chey I     16
Boeng Tompun I         15
Srah Chak              15
Veal Vong              14
Boeung Kak Ii          12
Nirouth                 9
Tuol Tompoung 1         9
Wat Phnom               7
Prampi Makara           6
Phsar Thmey I           6
Pur Senchey             4
Name: count, dtype: int64

## Create the reliable geocoding candidate table

In [93]:
reliable_geocode = location_df[
    location_df["geocode_level"].isin(
        ["project", "address"]
    )
].copy()

print(
    "Records:",
    len(reliable_geocode)
)

print(
    "Unique queries:",
    reliable_geocode[
        "geocode_query"
    ].nunique()
)

Records: 937
Unique queries: 156


In [94]:
reliable_geocode.groupby(
    "geocode_level"
).agg(
    records=("listing_id", "count"),
    unique_queries=("geocode_query", "nunique")
)

,records,unique_queries
geocode_level,,
address,498,79
project,439,77


## Inspect the 156 unique queries

In [95]:
unique_geocode_queries = (
    reliable_geocode[
        [
            "geocode_query",
            "geocode_level"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "geocode_level",
            "geocode_query"
        ]
    )
    .reset_index(drop=True)
)

unique_geocode_queries

,geocode_query,geocode_level
0,", BKK 1, Boeng Keng Kang, Phnom Penh, Boeung K...",address
1,", BKK 2, Boeng Keng Kang, Phnom Penh, Boeung K...",address
2,", BKK 3, Boeng Keng Kang, Phnom Penh, Boeung K...",address
3,", Boeung Kak 1, Toul Kork, Phnom Penh, Toul Ko...",address
4,", Boeung Kak 2, Toul Kork, Phnom Penh, Toul Ko...",address
...,...,...
151,"Urban Village Phase 2, Meanchey, Phnom Penh, C...",project
152,"Vista Condominium, Russey Keo, Phnom Penh, Cam...",project
153,"Vue Aston, Chbar Ampov, Phnom Penh, Cambodia",project
154,"Wealth Mansion, Chroy Changvar, Phnom Penh, Ca...",project


In [96]:
pd.set_option(
    "display.max_colwidth",
    None
)

unique_geocode_queries.head(50)

,geocode_query,geocode_level
0,", BKK 1, Boeng Keng Kang, Phnom Penh, Boeung Keng Kang, Phnom Penh, Cambodia",address
1,", BKK 2, Boeng Keng Kang, Phnom Penh, Boeung Keng Kang, Phnom Penh, Cambodia",address
2,", BKK 3, Boeng Keng Kang, Phnom Penh, Boeung Keng Kang, Phnom Penh, Cambodia",address
3,", Boeung Kak 1, Toul Kork, Phnom Penh, Toul Kork, Phnom Penh, Cambodia",address
4,", Boeung Kak 2, Toul Kork, Phnom Penh, Toul Kork, Phnom Penh, Cambodia",address
5,", Boeung Trabek, Chamkarmon, Phnom Penh, Chamkarmon, Phnom Penh, Cambodia",address
6,", Boeung Tumpun 1, Meanchey, Phnom Penh, Meanchey, Phnom Penh, Cambodia",address
7,", Boeung Tumpun 2, Meanchey, Phnom Penh, Meanchey, Phnom Penh, Cambodia",address
8,", Boeung Tumpun, Meanchey, Phnom Penh, Meanchey, Phnom Penh, Cambodia",address
9,", Chak Angrae Leu, Meanchey, Phnom Penh, Meanchey, Phnom Penh, Cambodia",address


## Check project queries separately

In [97]:
project_queries = (
    unique_geocode_queries[
        unique_geocode_queries[
            "geocode_level"
        ] == "project"
    ]
)

print(
    "Unique project queries:",
    len(project_queries)
)

project_queries.head(50)

Unique project queries: 77


,geocode_query,geocode_level
79,"Agile Sky Residence, Boeung Keng Kang, Phnom Penh, Cambodia",project
80,"Anata Residence, Meanchey, Phnom Penh, Cambodia",project
81,"Arakawa Residence, Sen Sok, Phnom Penh, Cambodia",project
82,"BK Residence, Prampi Makara, Phnom Penh, Cambodia",project
83,"Borey Peng Huoth: The Star Platinum Polaris I, Chbar Ampov, Phnom Penh, Cambodia",project
84,"CASA by Meridian, Chamkarmon, Phnom Penh, Cambodia",project
85,"Camko City, Russey Keo, Phnom Penh, Cambodia",project
86,"Chip Mong | Park Land TK Condo, Sen Sok, Phnom Penh, Cambodia",project
87,"Chip Mong | Park Land TK Condo, Toul Kork, Phnom Penh, Cambodia",project
88,"City View Residence, Daun Penh, Phnom Penh, Cambodia",project


## Check address queries

In [98]:
address_queries = (
    unique_geocode_queries[
        unique_geocode_queries[
            "geocode_level"
        ] == "address"
    ]
)

print(
    "Unique address queries:",
    len(address_queries)
)

address_queries.head(50)

Unique address queries: 79


,geocode_query,geocode_level
0,", BKK 1, Boeng Keng Kang, Phnom Penh, Boeung Keng Kang, Phnom Penh, Cambodia",address
1,", BKK 2, Boeng Keng Kang, Phnom Penh, Boeung Keng Kang, Phnom Penh, Cambodia",address
2,", BKK 3, Boeng Keng Kang, Phnom Penh, Boeung Keng Kang, Phnom Penh, Cambodia",address
3,", Boeung Kak 1, Toul Kork, Phnom Penh, Toul Kork, Phnom Penh, Cambodia",address
4,", Boeung Kak 2, Toul Kork, Phnom Penh, Toul Kork, Phnom Penh, Cambodia",address
5,", Boeung Trabek, Chamkarmon, Phnom Penh, Chamkarmon, Phnom Penh, Cambodia",address
6,", Boeung Tumpun 1, Meanchey, Phnom Penh, Meanchey, Phnom Penh, Cambodia",address
7,", Boeung Tumpun 2, Meanchey, Phnom Penh, Meanchey, Phnom Penh, Cambodia",address
8,", Boeung Tumpun, Meanchey, Phnom Penh, Meanchey, Phnom Penh, Cambodia",address
9,", Chak Angrae Leu, Meanchey, Phnom Penh, Meanchey, Phnom Penh, Cambodia",address


## validate the 77 project names

In [99]:
project_location_check = (
    location_df[
        location_df["project_name"].notna()
    ]
    .groupby("project_name")
    .agg(
        listings=("listing_id", "count"),
        district_count=("district", "nunique"),
        districts=(
            "district",
            lambda x: sorted(
                set(x.dropna())
            )
        )
    )
    .sort_values(
        ["district_count", "listings"],
        ascending=[False, False]
    )
)

project_location_check.head(30)

,listings,district_count,districts
project_name,,,
Chip Mong | Park Land TK Condo,15,2,"[Sen Sok, Toul Kork]"
PS Crystal Condominium,13,2,"[Chamkarmon, Meanchey]"
Urban Village Phase 2,59,1,[Meanchey]
Vue Aston,37,1,[Chbar Ampov]
Residence L Boeung Tompun,23,1,[Meanchey]
Morgan EnMaison | Condo Type,22,1,[Chroy Changvar]
Agile Sky Residence,15,1,[Boeung Keng Kang]
The Penthouse Residence,13,1,[Chamkarmon]
Time Square 306,11,1,[Boeung Keng Kang]


In [100]:
project_conflicts = (
    project_location_check[
        project_location_check[
            "district_count"
        ] > 1
    ]
)

print(
    "Projects with district conflicts:",
    len(project_conflicts)
)

project_conflicts

Projects with district conflicts: 2


,listings,district_count,districts
project_name,,,
Chip Mong | Park Land TK Condo,15,2,"[Sen Sok, Toul Kork]"
PS Crystal Condominium,13,2,"[Chamkarmon, Meanchey]"


## Create project-only queries

In [101]:
project_geocode = (
    location_df[
        location_df["project_name"].notna()
    ][
        [
            "listing_id",
            "project_name",
            "district"
        ]
    ]
    .copy()
)

In [102]:
# Build a cleaner query:

def build_project_query(row):

    project = str(
        row["project_name"]
    ).strip()

    if pd.notna(row["district"]):

        district = str(
            row["district"]
        ).strip()

        return (
            f"{project}, "
            f"{district}, "
            "Phnom Penh, Cambodia"
        )

    return (
        f"{project}, "
        "Phnom Penh, Cambodia"
    )


project_geocode[
    "project_geocode_query"
] = project_geocode.apply(
    build_project_query,
    axis=1
)

In [103]:
print(
    "Project records:",
    len(project_geocode)
)

print(
    "Unique project queries:",
    project_geocode[
        "project_geocode_query"
    ].nunique()
)

Project records: 439
Unique project queries: 77


## inspect the 28 conflicting listings

In [104]:
conflict_project_names = [
    "Chip Mong | Park Land TK Condo",
    "PS Crystal Condominium"
]

conflict_rows = location_df[
    location_df["project_name"].isin(
        conflict_project_names
    )
].copy()

conflict_rows[
    [
        "listing_id",
        "source",
        "title",
        "project_name",
        "district_original",
        "district",
        "district_recovery_source",
        "commune",
        "address",
        "location_text",
        "url"
    ]
].sort_values(
    [
        "project_name",
        "district"
    ]
)

,listing_id,source,title,project_name,district_original,district,district_recovery_source,commune,address,location_text,url
1,realestate.com.kh_259255,realestate.com.kh,2-Bedroom Condo for Sale at Chipmong Parkland TK Condo – High Floor | Fully Furnished,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/2-bed-2-bath-condo-259255/
11,realestate.com.kh_267317,realestate.com.kh,1-Bedroom Condo for Sale in Park Land TK Condo,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-267317/
83,realestate.com.kh_266220,realestate.com.kh,1-Bedroom Condo for Sale at Phnom Penh Thmei,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-266220/
113,realestate.com.kh_265677,realestate.com.kh,2-Bedroom Condo for Sale at Chipmong Parkland TK Condo | Fully Furnished,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/2-bed-2-bath-condo-265677/
158,realestate.com.kh_264705,realestate.com.kh,"1-Bedroom Condo for Sale at Chip Mong Parkland - $55,000",Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-264705/
178,realestate.com.kh_264303,realestate.com.kh,Fully Furnished 1-Bedroom Condo for Sale at The Parkland TK,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-264303/
276,realestate.com.kh_259657,realestate.com.kh,Modern 1-Bedroom Condo at Chip Mong Park Land TK Condo | City View,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259657/
283,realestate.com.kh_259533,realestate.com.kh,1-Bedroom Fully Furnished Condo at Chip Mong Parkland,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259533/
289,realestate.com.kh_259460,realestate.com.kh,1-Bedroom Fully Furnished Condo at Chip Mong Parkland,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259460/
297,realestate.com.kh_259307,realestate.com.kh,1 Bedroom - Fully Furnished Condo for Sale at Chip Mong Parkland,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259307/


## Also count the evidence

In [105]:
conflict_counts = (
    conflict_rows
    .groupby(
        [
            "project_name",
            "district"
        ]
    )
    .agg(
        listings=("listing_id", "count")
    )
    .reset_index()
)

conflict_counts

,project_name,district,listings
0,Chip Mong | Park Land TK Condo,Sen Sok,14
1,Chip Mong | Park Land TK Condo,Toul Kork,1
2,PS Crystal Condominium,Chamkarmon,1
3,PS Crystal Condominium,Meanchey,12


## Check original vs recovered

In [106]:
pd.crosstab(
    [
        conflict_rows["project_name"],
        conflict_rows["district"]
    ],
    conflict_rows[
        "district_recovery_source"
    ].fillna("original")
)

district_recovery_source                   original
project_name                   district            
Chip Mong | Park Land TK Condo Sen Sok           14
                               Toul Kork          1
PS Crystal Condominium         Chamkarmon         1
                               Meanchey          12

## Inspect each project separately

#### Project 1

In [107]:
conflict_rows[
    conflict_rows["project_name"]
    == "Chip Mong | Park Land TK Condo"
][
    [
        "listing_id",
        "title",
        "district_original",
        "district",
        "district_recovery_source",
        "commune",
        "location_text",
        "url"
    ]
]

,listing_id,title,district_original,district,district_recovery_source,commune,location_text,url
1,realestate.com.kh_259255,2-Bedroom Condo for Sale at Chipmong Parkland TK Condo – High Floor | Fully Furnished,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/2-bed-2-bath-condo-259255/
11,realestate.com.kh_267317,1-Bedroom Condo for Sale in Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-267317/
83,realestate.com.kh_266220,1-Bedroom Condo for Sale at Phnom Penh Thmei,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-266220/
113,realestate.com.kh_265677,2-Bedroom Condo for Sale at Chipmong Parkland TK Condo | Fully Furnished,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/2-bed-2-bath-condo-265677/
158,realestate.com.kh_264705,"1-Bedroom Condo for Sale at Chip Mong Parkland - $55,000",Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-264705/
178,realestate.com.kh_264303,Fully Furnished 1-Bedroom Condo for Sale at The Parkland TK,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-264303/
180,realestate.com.kh_264236,Fully Furnished 1-Bedroom Condo for Sale at The Parkland TK,Toul Kork,Toul Kork,None,Boeung Kak 1,", Boeung Kak 1, Toul Kork, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-264236/
276,realestate.com.kh_259657,Modern 1-Bedroom Condo at Chip Mong Park Land TK Condo | City View,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259657/
283,realestate.com.kh_259533,1-Bedroom Fully Furnished Condo at Chip Mong Parkland,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259533/
289,realestate.com.kh_259460,1-Bedroom Fully Furnished Condo at Chip Mong Parkland,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259460/


#### Project 2

In [108]:
conflict_rows[
    conflict_rows["project_name"]
    == "PS Crystal Condominium"
][
    [
        "listing_id",
        "title",
        "district_original",
        "district",
        "district_recovery_source",
        "commune",
        "location_text",
        "url"
    ]
]

,listing_id,title,district_original,district,district_recovery_source,commune,location_text,url
27,realestate.com.kh_267127,Cozy Studio Condo for Sale/ Rent Near BROWN Roastery 271,Meanchey,Meanchey,None,Boeung Tumpun,"Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/1-bed-1-bath-condo-267127/
52,realestate.com.kh_266779,Cozy unit for Sale /Rent at PS Crystal condominium,Meanchey,Meanchey,None,Boeung Tumpun,"Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/1-bed-1-bath-condo-266779/
130,realestate.com.kh_265212,1-Bedroom Condo (Corner) for Sale at PS Crystal Condo,Meanchey,Meanchey,None,Boeung Tumpun,"Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/1-bed-1-bath-condo-265212/
245,realestate.com.kh_262458,🏢 Condo for Sale – Boeng Tompun Area,Meanchey,Meanchey,None,Boeung Tumpun,", Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/1-bed-1-bath-condo-262458/
304,realestate.com.kh_259242,Modern 1-Bedroom Condo with Panoramic City Views – 7th Floor Corner Unit,Meanchey,Meanchey,None,Boeung Tumpun,"Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/1-bed-1-bath-condo-259242/
311,realestate.com.kh_259189,Corner Studio Unit for Sale in Beoung Tumpun,Meanchey,Meanchey,None,Boeung Tumpun,"Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/1-bed-1-bath-condo-259189/
421,realestate.com.kh_257016,A Studio Near BROWN Roastery 271 – Fully Furnished with Balcony,Meanchey,Meanchey,None,Boeung Tumpun,"Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/1-bed-1-bath-studio-257016/
422,realestate.com.kh_256995,2 Bedrooms Condominium For Sale,Meanchey,Meanchey,None,Boeung Tumpun,", Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/2-bed-2-bath-condo-256995/
511,realestate.com.kh_251229,Fully furnished Condo for Rent I PS Crystal condominium,Meanchey,Meanchey,None,Boeung Tumpun,", Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/1-bed-1-bath-condo-251229/
682,realestate.com.kh_241107,Studio Room for Sale/Rent in PS Crystal Condominium,Meanchey,Meanchey,None,Boeung Tumpun,"Boeung Tumpun, Meanchey, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/1-bed-1-bath-studio-241107/


## Now we need the second result: detailed conflict rows

In [109]:
conflict_rows[
    [
        "listing_id",
        "source",
        "title",
        "project_name",
        "district_original",
        "district",
        "district_recovery_source",
        "commune",
        "address",
        "location_text",
        "url"
    ]
].sort_values(
    [
        "project_name",
        "district"
    ]
)

,listing_id,source,title,project_name,district_original,district,district_recovery_source,commune,address,location_text,url
1,realestate.com.kh_259255,realestate.com.kh,2-Bedroom Condo for Sale at Chipmong Parkland TK Condo – High Floor | Fully Furnished,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/2-bed-2-bath-condo-259255/
11,realestate.com.kh_267317,realestate.com.kh,1-Bedroom Condo for Sale in Park Land TK Condo,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-267317/
83,realestate.com.kh_266220,realestate.com.kh,1-Bedroom Condo for Sale at Phnom Penh Thmei,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-266220/
113,realestate.com.kh_265677,realestate.com.kh,2-Bedroom Condo for Sale at Chipmong Parkland TK Condo | Fully Furnished,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/2-bed-2-bath-condo-265677/
158,realestate.com.kh_264705,realestate.com.kh,"1-Bedroom Condo for Sale at Chip Mong Parkland - $55,000",Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-264705/
178,realestate.com.kh_264303,realestate.com.kh,Fully Furnished 1-Bedroom Condo for Sale at The Parkland TK,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-264303/
276,realestate.com.kh_259657,realestate.com.kh,Modern 1-Bedroom Condo at Chip Mong Park Land TK Condo | City View,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259657/
283,realestate.com.kh_259533,realestate.com.kh,1-Bedroom Fully Furnished Condo at Chip Mong Parkland,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259533/
289,realestate.com.kh_259460,realestate.com.kh,1-Bedroom Fully Furnished Condo at Chip Mong Parkland,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,", Phnom Penh Thmey, Sen Sok, Phnom Penh",", Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259460/
297,realestate.com.kh_259307,realestate.com.kh,1 Bedroom - Fully Furnished Condo for Sale at Chip Mong Parkland,Chip Mong | Park Land TK Condo,Sen Sok,Sen Sok,None,Phnom Penh Thmey,"Phnom Penh Thmey, Sen Sok, Phnom Penh","Phnom Penh Thmey, Sen Sok, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-259307/


In [110]:
minority_conflicts = conflict_rows[
    (
        (conflict_rows["project_name"]
         == "Chip Mong | Park Land TK Condo")
        &
        (conflict_rows["district"]
         == "Toul Kork")
    )
    |
    (
        (conflict_rows["project_name"]
         == "PS Crystal Condominium")
        &
        (conflict_rows["district"]
         == "Chamkarmon")
    )
]

minority_conflicts[
    [
        "listing_id",
        "source",
        "title",
        "project_name",
        "district",
        "commune",
        "address",
        "location_text",
        "url"
    ]
]

,listing_id,source,title,project_name,district,commune,address,location_text,url
180,realestate.com.kh_264236,realestate.com.kh,Fully Furnished 1-Bedroom Condo for Sale at The Parkland TK,Chip Mong | Park Land TK Condo,Toul Kork,Boeung Kak 1,", Boeung Kak 1, Toul Kork, Phnom Penh",", Boeung Kak 1, Toul Kork, Phnom Penh",https://www.realestate.com.kh/new-developments/chip-mong-park-land-tk-condo/1-bed-1-bath-condo-264236/
738,realestate.com.kh_142696,realestate.com.kh,Modern Apartment for Sale Near Toul Tom Poung Market,PS Crystal Condominium,Chamkarmon,Toul Tum Poung 2,"Toul Tum Poung 2, Chamkarmon, Phnom Penh","Toul Tum Poung 2, Chamkarmon, Phnom Penh",https://www.realestate.com.kh/new-developments/ps-crystal-condominium/2-bed-2-bath-condo-142696/


In [111]:
project_conflict_fixes = {
    "realestate.com.kh_264236": "Sen Sok",
    "realestate.com.kh_142696": "Meanchey",
}

for listing_id, corrected_district in project_conflict_fixes.items():

    mask = (
        location_df["listing_id"]
        == listing_id
    )

    location_df.loc[
        mask,
        "district"
    ] = corrected_district

    location_df.loc[
        mask,
        "district_recovery_source"
    ] = "project_location_verification"

In [112]:
project_location_check = (
    location_df[
        location_df["project_name"].notna()
    ]
    .groupby("project_name")
    .agg(
        listings=("listing_id", "count"),
        district_count=("district", "nunique"),
        districts=(
            "district",
            lambda x: sorted(
                set(x.dropna())
            )
        )
    )
)

project_conflicts = project_location_check[
    project_location_check[
        "district_count"
    ] > 1
]

print(
    "Projects with district conflicts:",
    len(project_conflicts)
)

project_conflicts

Projects with district conflicts: 0


,listings,district_count,districts
project_name,,,


In [113]:
location_df.to_csv(
    "../data/gold/"
    "property_listings_location_enriched.csv",
    index=False,
    encoding="utf-8-sig"
)

print(location_df.shape)

(2673, 31)


## geocode only the project locations

We decided the safest first group is the 439 listings with known project_name, representing about 77 unique projects.

In [114]:
def build_project_query(row):
    project = str(row["project_name"]).strip()

    if pd.notna(row["district"]):
        district = str(row["district"]).strip()

        return (
            f"{project}, "
            f"{district}, "
            "Phnom Penh, Cambodia"
        )

    return (
        f"{project}, "
        "Phnom Penh, Cambodia"
    )

In [115]:
project_df = location_df[
    location_df["project_name"].notna()
].copy()

project_df["project_geocode_query"] = (
    project_df.apply(
        build_project_query,
        axis=1
    )
)

print("Project records:", len(project_df))

print(
    "Unique project queries:",
    project_df[
        "project_geocode_query"
    ].nunique()
)

Project records: 439
Unique project queries: 75


## Create the 75-row geocoding table

In [116]:
project_queries = (
    project_df[
        [
            "project_name",
            "district",
            "project_geocode_query"
        ]
    ]
    .drop_duplicates(
        subset=["project_geocode_query"]
    )
    .reset_index(drop=True)
)

print(project_queries.shape)

project_queries.head(20)

(75, 3)


,project_name,district,project_geocode_query
0,Time Square II,Toul Kork,"Time Square II, Toul Kork, Phnom Penh, Cambodia"
1,Chip Mong | Park Land TK Condo,Sen Sok,"Chip Mong | Park Land TK Condo, Sen Sok, Phnom Penh, Cambodia"
2,Time Square 306,Boeung Keng Kang,"Time Square 306, Boeung Keng Kang, Phnom Penh, Cambodia"
3,One Park,Daun Penh,"One Park, Daun Penh, Phnom Penh, Cambodia"
4,TK Star International,Toul Kork,"TK Star International, Toul Kork, Phnom Penh, Cambodia"
5,Urban Village Phase 2,Meanchey,"Urban Village Phase 2, Meanchey, Phnom Penh, Cambodia"
6,The Garden Residency II,Sen Sok,"The Garden Residency II, Sen Sok, Phnom Penh, Cambodia"
7,Time Square 3,Toul Kork,"Time Square 3, Toul Kork, Phnom Penh, Cambodia"
8,Anata Residence,Meanchey,"Anata Residence, Meanchey, Phnom Penh, Cambodia"
9,Time Square 302,Boeung Keng Kang,"Time Square 302, Boeung Keng Kang, Phnom Penh, Cambodia"


This is the table we geocode.

## Use OpenStreetMap Nominatim

For this one-time student-project task, Nominatim can be used deliberately for a small batch, but its public service has strict rules: maximum 1 request/second, a meaningful User-Agent, one thread, and results should be cached locally.

In [117]:
import requests
import time
import pandas as pd
from pathlib import Path

## Geocoding code

Use a clear User-Agent identifying project:

In [118]:
USER_AGENT = (
    "pp-propertylens-student-project/1.0 "
    "(Phnom Penh real estate research)"
)

In [119]:
# Create output path

cache_path = Path(
    "../data/geo/project_geocode_results.csv"
)

In [120]:
def geocode_project(query):

    url = (
        "https://nominatim.openstreetmap.org/search"
    )

    params = {
        "q": query,
        "format": "jsonv2",
        "limit": 1,
        "addressdetails": 1,
        "countrycodes": "kh",
    }

    headers = {
        "User-Agent": USER_AGENT
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    results = response.json()

    if not results:
        return {
            "latitude": None,
            "longitude": None,
            "display_name": None,
            "osm_type": None,
            "osm_id": None,
            "geocode_status": "not_found",
        }

    result = results[0]

    return {
        "latitude": float(result["lat"]),
        "longitude": float(result["lon"]),
        "display_name": result.get(
            "display_name"
        ),
        "osm_type": result.get(
            "osm_type"
        ),
        "osm_id": result.get(
            "osm_id"
        ),
        "geocode_status": "found",
    }

## Run the 75 queries safely

In [121]:
geocode_results = []

for i, row in project_queries.iterrows():

    query = row[
        "project_geocode_query"
    ]

    print(
        f"{i + 1}/{len(project_queries)} "
        f"{query}"
    )

    try:
        result = geocode_project(
            query
        )

    except Exception as e:

        result = {
            "latitude": None,
            "longitude": None,
            "display_name": None,
            "osm_type": None,
            "osm_id": None,
            "geocode_status": f"error: {e}",
        }

    geocode_results.append({
        "project_name": row[
            "project_name"
        ],
        "district": row[
            "district"
        ],
        "project_geocode_query": query,
        **result,
    })

    # Respect public Nominatim rate limits.
    time.sleep(1.2)

1/75 Time Square II, Toul Kork, Phnom Penh, Cambodia
2/75 Chip Mong | Park Land TK Condo, Sen Sok, Phnom Penh, Cambodia
3/75 Time Square 306, Boeung Keng Kang, Phnom Penh, Cambodia
4/75 One Park, Daun Penh, Phnom Penh, Cambodia
5/75 TK Star International, Toul Kork, Phnom Penh, Cambodia
6/75 Urban Village Phase 2, Meanchey, Phnom Penh, Cambodia
7/75 The Garden Residency II, Sen Sok, Phnom Penh, Cambodia
8/75 Time Square 3, Toul Kork, Phnom Penh, Cambodia
9/75 Anata Residence, Meanchey, Phnom Penh, Cambodia
10/75 Time Square 302, Boeung Keng Kang, Phnom Penh, Cambodia
11/75 Time Square 8, Chamkarmon, Phnom Penh, Cambodia
12/75 J Tower 2 Condominium, Boeung Keng Kang, Phnom Penh, Cambodia
13/75 North Park Condominium, Sen Sok, Phnom Penh, Cambodia
14/75 Yuetai Phnom Penh Harbour, Daun Penh, Phnom Penh, Cambodia
15/75 Vue Aston, Chbar Ampov, Phnom Penh, Cambodia
16/75 PS Crystal Condominium, Meanchey, Phnom Penh, Cambodia
17/75 Agile Sky Residence, Boeung Keng Kang, Phnom Penh, Cambodia
1

The public Nominatim policy caps ordinary usage at one request per second, so don't remove that delay or run this in parallel.

## Save immediately

In [122]:
project_geocode_results = pd.DataFrame(
    geocode_results
)

project_geocode_results.to_csv(
    cache_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    project_geocode_results[
        "geocode_status"
    ].value_counts()
)

geocode_status
not_found    59
found        16
Name: count, dtype: int64


In [123]:
project_geocode_results.head(20)

,project_name,district,project_geocode_query,latitude,longitude,display_name,osm_type,osm_id,geocode_status
0,Time Square II,Toul Kork,"Time Square II, Toul Kork, Phnom Penh, Cambodia",NaN,NaN,NaN,NaN,NaN,not_found
1,Chip Mong | Park Land TK Condo,Sen Sok,"Chip Mong | Park Land TK Condo, Sen Sok, Phnom Penh, Cambodia",NaN,NaN,NaN,NaN,NaN,not_found
2,Time Square 306,Boeung Keng Kang,"Time Square 306, Boeung Keng Kang, Phnom Penh, Cambodia",NaN,NaN,NaN,NaN,NaN,not_found
3,One Park,Daun Penh,"One Park, Daun Penh, Phnom Penh, Cambodia",11.576143,104.905704,"One Park, សហគមន៍​បឹងកក់, សង្កាត់ផ្សារថ្មីទី ២, ខណ្ឌដូនពេញ, រាជធានីភ្នំពេញ, 120210, ព្រះរាជាណាចក្រ​កម្ពុជា",way,3.479344e+08,found
4,TK Star International,Toul Kork,"TK Star International, Toul Kork, Phnom Penh, Cambodia",NaN,NaN,NaN,NaN,NaN,not_found
5,Urban Village Phase 2,Meanchey,"Urban Village Phase 2, Meanchey, Phnom Penh, Cambodia",NaN,NaN,NaN,NaN,NaN,not_found
6,The Garden Residency II,Sen Sok,"The Garden Residency II, Sen Sok, Phnom Penh, Cambodia",NaN,NaN,NaN,NaN,NaN,not_found
7,Time Square 3,Toul Kork,"Time Square 3, Toul Kork, Phnom Penh, Cambodia",11.583571,104.897168,"Time Square 3, ផ្លូវ ៣១៧, សង្កាត់បឹងកក់ទី ១, ខណ្ឌទួលគោក, រាជធានីភ្នំពេញ, 120407, ព្រះរាជាណាចក្រ​កម្ពុជា",way,1.296932e+09,found
8,Anata Residence,Meanchey,"Anata Residence, Meanchey, Phnom Penh, Cambodia",NaN,NaN,NaN,NaN,NaN,not_found
9,Time Square 302,Boeung Keng Kang,"Time Square 302, Boeung Keng Kang, Phnom Penh, Cambodia",NaN,NaN,NaN,NaN,NaN,not_found


## Very important: don't map coordinates back yet

Even when Nominatim says:

found

we should first validate the returned result.

For example:

Query:
Picasso Sky Garden, BKK, Phnom Penh

Returned:
Picasso City Garden, another country

would be wrong.

In [124]:
# after geocoding, check

project_geocode_results[
    [
        "project_name",
        "district",
        "latitude",
        "longitude",
        "display_name",
        "geocode_status"
    ]
]

,project_name,district,latitude,longitude,display_name,geocode_status
0,Time Square II,Toul Kork,NaN,NaN,NaN,not_found
1,Chip Mong | Park Land TK Condo,Sen Sok,NaN,NaN,NaN,not_found
2,Time Square 306,Boeung Keng Kang,NaN,NaN,NaN,not_found
3,One Park,Daun Penh,11.576143,104.905704,"One Park, សហគមន៍​បឹងកក់, សង្កាត់ផ្សារថ្មីទី ២, ខណ្ឌដូនពេញ, រាជធានីភ្នំពេញ, 120210, ព្រះរាជាណាចក្រ​កម្ពុជា",found
4,TK Star International,Toul Kork,NaN,NaN,NaN,not_found
...,...,...,...,...,...,...
70,Grand Condo 7,Chroy Changvar,NaN,NaN,NaN,not_found
71,Sen Sok Town,Sen Sok,NaN,NaN,NaN,not_found
72,Star City,Sen Sok,11.562060,104.868656,"Star City, ភូមិចុងថ្នល់ខាងលិច, សង្កាត់ទឹកថ្លា, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120802, ព្រះរាជាណាចក្រ​កម្ពុជា",found
73,Camko City,Russey Keo,11.594865,104.897716,"Camko City R2, ភូមិទួលថ្ងាន់, សង្កាត់ទួលសង្កែទី២, ខណ្ឌឫស្សីកែវ, រាជធានីភ្នំពេញ, 120707, ព្រះរាជាណាចក្រ​កម្ពុជា",found


In [125]:
print(
    "Found:",
    (
        project_geocode_results[
            "geocode_status"
        ] == "found"
    ).sum()
)

print(
    "Not found:",
    (
        project_geocode_results[
            "geocode_status"
        ] == "not_found"
    ).sum()
)

Found: 16
Not found: 59


In [126]:
project_geocode_results[
    "geocode_status"
].value_counts(
    dropna=False
)

geocode_status
not_found    59
found        16
Name: count, dtype: int64

## validate the 16 found projects 📍

Do not merge them back into the 2,673 listings yet.

In [127]:
found_projects = project_geocode_results[
    project_geocode_results[
        "geocode_status"
    ] == "found"
].copy()

found_projects[
    [
        "project_name",
        "district",
        "project_geocode_query",
        "latitude",
        "longitude",
        "display_name"
    ]
]

,project_name,district,project_geocode_query,latitude,longitude,display_name
3,One Park,Daun Penh,"One Park, Daun Penh, Phnom Penh, Cambodia",11.576143,104.905704,"One Park, សហគមន៍​បឹងកក់, សង្កាត់ផ្សារថ្មីទី ២, ខណ្ឌដូនពេញ, រាជធានីភ្នំពេញ, 120210, ព្រះរាជាណាចក្រ​កម្ពុជា"
7,Time Square 3,Toul Kork,"Time Square 3, Toul Kork, Phnom Penh, Cambodia",11.583571,104.897168,"Time Square 3, ផ្លូវ ៣១៧, សង្កាត់បឹងកក់ទី ១, ខណ្ឌទួលគោក, រាជធានីភ្នំពេញ, 120407, ព្រះរាជាណាចក្រ​កម្ពុជា"
12,North Park Condominium,Sen Sok,"North Park Condominium, Sen Sok, Phnom Penh, Cambodia",11.551292,104.872536,"ផ្លូវចេញ ណស៏ផាក ខុនដូមីនៀម, ភូមិស្លែងរលើង, សង្កាត់អូរបែកក្អម, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120805, ព្រះរាជាណាចក្រ​កម្ពុជា"
13,Yuetai Phnom Penh Harbour,Daun Penh,"Yuetai Phnom Penh Harbour, Daun Penh, Phnom Penh, Cambodia",11.581475,104.922626,"Yuetai Phnom Penh Harbour City, សហគមន៍​បឹងកក់, សង្កាត់ស្រះចក, ខណ្ឌដូនពេញ, រាជធានីភ្នំពេញ, 120210, ព្រះរាជាណាចក្រ​កម្ពុជា"
18,Arakawa Residence,Sen Sok,"Arakawa Residence, Sen Sok, Phnom Penh, Cambodia",11.559708,104.880765,"លំនៅឋានអារ៉ាខាវ៉ា, ភូមិផ្សារទឹកថ្លា, សង្កាត់ទឹកថ្លា, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120802, ព្រះរាជាណាចក្រ​កម្ពុជា"
31,Residence H Sen Sok,Sen Sok,"Residence H Sen Sok, Sen Sok, Phnom Penh, Cambodia",11.559708,104.880765,"លំនៅឋានអារ៉ាខាវ៉ា, ភូមិផ្សារទឹកថ្លា, សង្កាត់ទឹកថ្លា, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120802, ព្រះរាជាណាចក្រ​កម្ពុជា"
35,Olympia City,Prampi Makara,"Olympia City, Prampi Makara, Phnom Penh, Cambodia",11.560738,104.913245,"Olympia City, សង្កាត់អូរឬស្សីទី ២, ប្រាំពីរមករា, រាជធានីភ្នំពេញ, 120302, ព្រះរាជាណាចក្រ​កម្ពុជា"
37,The Garden Residency,Sen Sok,"The Garden Residency, Sen Sok, Phnom Penh, Cambodia",11.574445,104.881375,"The Garden Residency, ភូមិភ្នំពេញថ្មី, សង្កាត់ភ្នំពេញថ្មី, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120801, ព្រះរាជាណាចក្រ​កម្ពុជា"
43,Wealth Mansion,Chroy Changvar,"Wealth Mansion, Chroy Changvar, Phnom Penh, Cambodia",11.585354,104.927542,"Wealth Mansion, ផ្លូវ ទន្លេសាប, សង្កាត់ជ្រោយចង្វា, ខណ្ឌជ្រោយចង្វារ, រាជធានីភ្នំពេញ, 121001, ព្រះរាជាណាចក្រ​កម្ពុជា"
47,La Vista One,Chroy Changvar,"La Vista One, Chroy Changvar, Phnom Penh, Cambodia",11.581297,104.938060,"La Vista One, ផ្លូវ ទន្លេមេគង្គ, ចុងកោះមានជ័យបារមីតេជោ, សង្កាត់ជ្រោយចង្វា, ខណ្ឌជ្រោយចង្វារ, រាជធានីភ្នំពេញ, 121001, ព្រះរាជាណាចក្រ​កម្ពុជា"


For each of the 16, check:

Project name matches?       

Returned place in Cambodia? 

Returned place in Phnom Penh? 

District/location plausible? 

## Add a validation column

In [128]:
found_projects[
    "manual_valid"
] = ""

found_projects[
    "validation_note"
] = ""

## After validating the 16

Then we can make a second-pass search for the 59 not-found projects using a simpler query.

Your first query is:

Project Name + District + Phnom Penh + Cambodia

For the retry, use:

Project Name + Phnom Penh + Cambodia

Sometimes district wording prevents a match.

In [129]:
not_found_projects = (
    project_geocode_results[
        project_geocode_results[
            "geocode_status"
        ] == "not_found"
    ]
    .copy()
)

In [130]:
not_found_projects[
    "retry_query"
] = (
    not_found_projects[
        "project_name"
    ].astype(str)
    + ", Phnom Penh, Cambodia"
)

In [131]:
not_found_projects[
    [
        "project_name",
        "project_geocode_query",
        "retry_query"
    ]
].head(20)

,project_name,project_geocode_query,retry_query
0,Time Square II,"Time Square II, Toul Kork, Phnom Penh, Cambodia","Time Square II, Phnom Penh, Cambodia"
1,Chip Mong | Park Land TK Condo,"Chip Mong | Park Land TK Condo, Sen Sok, Phnom Penh, Cambodia","Chip Mong | Park Land TK Condo, Phnom Penh, Cambodia"
2,Time Square 306,"Time Square 306, Boeung Keng Kang, Phnom Penh, Cambodia","Time Square 306, Phnom Penh, Cambodia"
4,TK Star International,"TK Star International, Toul Kork, Phnom Penh, Cambodia","TK Star International, Phnom Penh, Cambodia"
5,Urban Village Phase 2,"Urban Village Phase 2, Meanchey, Phnom Penh, Cambodia","Urban Village Phase 2, Phnom Penh, Cambodia"
6,The Garden Residency II,"The Garden Residency II, Sen Sok, Phnom Penh, Cambodia","The Garden Residency II, Phnom Penh, Cambodia"
8,Anata Residence,"Anata Residence, Meanchey, Phnom Penh, Cambodia","Anata Residence, Phnom Penh, Cambodia"
9,Time Square 302,"Time Square 302, Boeung Keng Kang, Phnom Penh, Cambodia","Time Square 302, Phnom Penh, Cambodia"
10,Time Square 8,"Time Square 8, Chamkarmon, Phnom Penh, Cambodia","Time Square 8, Phnom Penh, Cambodia"
11,J Tower 2 Condominium,"J Tower 2 Condominium, Boeung Keng Kang, Phnom Penh, Cambodia","J Tower 2 Condominium, Phnom Penh, Cambodia"


## Check for duplicate coordinates

In [133]:
duplicate_coordinates = (
    found_projects[
        found_projects.duplicated(
            subset=[
                "latitude",
                "longitude"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "latitude",
            "longitude"
        ]
    )
)

duplicate_coordinates[
    [
        "project_name",
        "district",
        "latitude",
        "longitude",
        "display_name"
    ]
]

,project_name,district,latitude,longitude,display_name
18,Arakawa Residence,Sen Sok,11.559708,104.880765,"លំនៅឋានអារ៉ាខាវ៉ា, ភូមិផ្សារទឹកថ្លា, សង្កាត់ទឹកថ្លា, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120802, ព្រះរាជាណាចក្រ​កម្ពុជា"
31,Residence H Sen Sok,Sen Sok,11.559708,104.880765,"លំនៅឋានអារ៉ាខាវ៉ា, ភូមិផ្សារទឹកថ្លា, សង្កាត់ទឹកថ្លា, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120802, ព្រះរាជាណាចក្រ​កម្ពុជា"


This tells us whether multiple project names were mapped to exactly the same point.

## Validate that coordinates are inside Phnom Penh

All of property projects should be in Phnom Penh, so we can do a simple geographic sanity check.

Use a broad Phnom Penh bounding box:

In [134]:
def inside_phnom_penh(lat, lon):
    return (
        11.35 <= lat <= 11.75
        and
        104.70 <= lon <= 105.10
    )

In [135]:
found_projects[
    "inside_phnom_penh"
] = found_projects.apply(
    lambda row: inside_phnom_penh(
        row["latitude"],
        row["longitude"]
    ),
    axis=1
)

In [136]:
found_projects[
    "inside_phnom_penh"
].value_counts()

inside_phnom_penh
True    16
Name: count, dtype: int64

## Create validation status

In [137]:
found_projects[
    "coordinate_validation"
] = "valid"

In [138]:
# Mark duplicate coordinate cases:

duplicate_mask = (
    found_projects.duplicated(
        subset=[
            "latitude",
            "longitude"
        ],
        keep=False
    )
)

found_projects.loc[
    duplicate_mask,
    "coordinate_validation"
] = "review_duplicate_coordinate"

In [139]:
# Mark anything outside Phnom Penh:

found_projects.loc[
    ~found_projects[
        "inside_phnom_penh"
    ],
    "coordinate_validation"
] = "invalid_outside_phnom_penh"

In [140]:
found_projects[
    "coordinate_validation"
].value_counts()

coordinate_validation
valid                          14
review_duplicate_coordinate     2
Name: count, dtype: int64

In [141]:
found_projects[
    [
        "project_name",
        "district",
        "latitude",
        "longitude",
        "display_name",
        "coordinate_validation"
    ]
]

,project_name,district,latitude,longitude,display_name,coordinate_validation
3,One Park,Daun Penh,11.576143,104.905704,"One Park, សហគមន៍​បឹងកក់, សង្កាត់ផ្សារថ្មីទី ២, ខណ្ឌដូនពេញ, រាជធានីភ្នំពេញ, 120210, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
7,Time Square 3,Toul Kork,11.583571,104.897168,"Time Square 3, ផ្លូវ ៣១៧, សង្កាត់បឹងកក់ទី ១, ខណ្ឌទួលគោក, រាជធានីភ្នំពេញ, 120407, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
12,North Park Condominium,Sen Sok,11.551292,104.872536,"ផ្លូវចេញ ណស៏ផាក ខុនដូមីនៀម, ភូមិស្លែងរលើង, សង្កាត់អូរបែកក្អម, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120805, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
13,Yuetai Phnom Penh Harbour,Daun Penh,11.581475,104.922626,"Yuetai Phnom Penh Harbour City, សហគមន៍​បឹងកក់, សង្កាត់ស្រះចក, ខណ្ឌដូនពេញ, រាជធានីភ្នំពេញ, 120210, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
18,Arakawa Residence,Sen Sok,11.559708,104.880765,"លំនៅឋានអារ៉ាខាវ៉ា, ភូមិផ្សារទឹកថ្លា, សង្កាត់ទឹកថ្លា, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120802, ព្រះរាជាណាចក្រ​កម្ពុជា",review_duplicate_coordinate
31,Residence H Sen Sok,Sen Sok,11.559708,104.880765,"លំនៅឋានអារ៉ាខាវ៉ា, ភូមិផ្សារទឹកថ្លា, សង្កាត់ទឹកថ្លា, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120802, ព្រះរាជាណាចក្រ​កម្ពុជា",review_duplicate_coordinate
35,Olympia City,Prampi Makara,11.560738,104.913245,"Olympia City, សង្កាត់អូរឬស្សីទី ២, ប្រាំពីរមករា, រាជធានីភ្នំពេញ, 120302, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
37,The Garden Residency,Sen Sok,11.574445,104.881375,"The Garden Residency, ភូមិភ្នំពេញថ្មី, សង្កាត់ភ្នំពេញថ្មី, ខណ្ឌសែនសុខ, រាជធានីភ្នំពេញ, 120801, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
43,Wealth Mansion,Chroy Changvar,11.585354,104.927542,"Wealth Mansion, ផ្លូវ ទន្លេសាប, សង្កាត់ជ្រោយចង្វា, ខណ្ឌជ្រោយចង្វារ, រាជធានីភ្នំពេញ, 121001, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
47,La Vista One,Chroy Changvar,11.581297,104.938060,"La Vista One, ផ្លូវ ទន្លេមេគង្គ, ចុងកោះមានជ័យបារមីតេជោ, សង្កាត់ជ្រោយចង្វា, ខណ្ឌជ្រោយចង្វារ, រាជធានីភ្នំពេញ, 121001, ព្រះរាជាណាចក្រ​កម្ពុជា",valid


## Save the validation result

In [142]:
found_projects.to_csv(
    "../data/geo/"
    "project_geocode_found_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

## Mark the first pass result

In [146]:
found_projects["coordinate_validation"] = "valid_first_pass"

found_projects.loc[
    found_projects["project_name"]
    == "Residence H Sen Sok",
    "coordinate_validation"
] = "retry_wrong_match"

In [147]:
# check

found_projects[
    "coordinate_validation"
].value_counts()

coordinate_validation
valid_first_pass     15
retry_wrong_match     1
Name: count, dtype: int64

## Retry Residence H with a more specific query

In [148]:
residence_h_query = (
    "Residence H Sen Sok, "
    "Phnom Penh Thmey, Sen Sok, "
    "Phnom Penh, Cambodia"
)

result = geocode_project(
    residence_h_query
)

result

{'latitude': None,
 'longitude': None,
 'display_name': None,
 'osm_type': None,
 'osm_id': None,
 'geocode_status': 'not_found'}

## Retry the 59 not-found projects

In [149]:
not_found_projects = (
    project_geocode_results[
        project_geocode_results[
            "geocode_status"
        ] == "not_found"
    ]
    .copy()
)

In [150]:
# create simpler query for retrying

not_found_projects[
    "retry_query"
] = (
    not_found_projects[
        "project_name"
    ].astype(str).str.strip()
    + ", Phnom Penh, Cambodia"
)

In [151]:
# inspect the retry queries

not_found_projects[
    [
        "project_name",
        "district",
        "retry_query"
    ]
].head(20)

,project_name,district,retry_query
0,Time Square II,Toul Kork,"Time Square II, Phnom Penh, Cambodia"
1,Chip Mong | Park Land TK Condo,Sen Sok,"Chip Mong | Park Land TK Condo, Phnom Penh, Cambodia"
2,Time Square 306,Boeung Keng Kang,"Time Square 306, Phnom Penh, Cambodia"
4,TK Star International,Toul Kork,"TK Star International, Phnom Penh, Cambodia"
5,Urban Village Phase 2,Meanchey,"Urban Village Phase 2, Phnom Penh, Cambodia"
6,The Garden Residency II,Sen Sok,"The Garden Residency II, Phnom Penh, Cambodia"
8,Anata Residence,Meanchey,"Anata Residence, Phnom Penh, Cambodia"
9,Time Square 302,Boeung Keng Kang,"Time Square 302, Phnom Penh, Cambodia"
10,Time Square 8,Chamkarmon,"Time Square 8, Phnom Penh, Cambodia"
11,J Tower 2 Condominium,Boeung Keng Kang,"J Tower 2 Condominium, Phnom Penh, Cambodia"


In [152]:
retry_results = []

for i, row in not_found_projects.iterrows():

    query = row["retry_query"]

    print(
        f"{len(retry_results) + 1}/"
        f"{len(not_found_projects)} "
        f"{query}"
    )

    try:
        result = geocode_project(
            query
        )

    except Exception as e:
        result = {
            "latitude": None,
            "longitude": None,
            "display_name": None,
            "osm_type": None,
            "osm_id": None,
            "geocode_status": f"error: {e}",
        }

    retry_results.append({
        "project_name": row["project_name"],
        "district": row["district"],
        "retry_query": query,
        **result,
    })

    time.sleep(1.2)

1/59 Time Square II, Phnom Penh, Cambodia
2/59 Chip Mong | Park Land TK Condo, Phnom Penh, Cambodia
3/59 Time Square 306, Phnom Penh, Cambodia
4/59 TK Star International, Phnom Penh, Cambodia
5/59 Urban Village Phase 2, Phnom Penh, Cambodia
6/59 The Garden Residency II, Phnom Penh, Cambodia
7/59 Anata Residence, Phnom Penh, Cambodia
8/59 Time Square 302, Phnom Penh, Cambodia
9/59 Time Square 8, Phnom Penh, Cambodia
10/59 J Tower 2 Condominium, Phnom Penh, Cambodia
11/59 Vue Aston, Phnom Penh, Cambodia
12/59 PS Crystal Condominium, Phnom Penh, Cambodia
13/59 Agile Sky Residence, Phnom Penh, Cambodia
14/59 Residence L Boeung Tompun, Phnom Penh, Cambodia
15/59 Morgan EnMaison | Condo Type, Phnom Penh, Cambodia
16/59 M Residence, Phnom Penh, Cambodia
17/59 The Penthouse Residence, Phnom Penh, Cambodia
18/59 L Residence Boeung Trabek II, Phnom Penh, Cambodia
19/59 L Residence Borei Keila, Phnom Penh, Cambodia
20/59 Urban Village Phase 1, Phnom Penh, Cambodia
21/59 J Tower 3 Condominium, Phn

In [153]:
# Save retry results

retry_results_df = pd.DataFrame(
    retry_results
)

retry_results_df.to_csv(
    "../data/geo/project_geocode_retry_results.csv",
    index=False,
    encoding="utf-8-sig"
)

In [154]:
# Finally check:

retry_results_df[
    "geocode_status"
].value_counts()

geocode_status
not_found    44
found        15
Name: count, dtype: int64

## validate the 15 newly found projects

In [155]:
retry_found = retry_results_df[
    retry_results_df["geocode_status"] == "found"
].copy()

print("Retry found:", len(retry_found))

Retry found: 15


In [156]:
# Inspect 

retry_found[
    [
        "project_name",
        "district",
        "retry_query",
        "latitude",
        "longitude",
        "display_name"
    ]
]

,project_name,district,retry_query,latitude,longitude,display_name
2,Time Square 306,Boeung Keng Kang,"Time Square 306, Phnom Penh, Cambodia",11.555312,104.923949,"Time Square Hotel & Apartment, No 9. E1, ផ្លូវ ២៧៨, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា"
7,Time Square 302,Boeung Keng Kang,"Time Square 302, Phnom Penh, Cambodia",11.555312,104.923949,"Time Square Hotel & Apartment, No 9. E1, ផ្លូវ ២៧៨, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា"
10,Vue Aston,Chbar Ampov,"Vue Aston, Phnom Penh, Cambodia",11.546288,104.946528,"Vue Aston, Tonle Bassac Promenade, កោះពេជ្រ, សង្កាត់ទន្លេបាសាក់, ខណ្ឌចំការមន, រាជធានីភ្នំពេញ, 120101, ព្រះរាជាណាចក្រ​កម្ពុជា"
12,Agile Sky Residence,Boeung Keng Kang,"Agile Sky Residence, Phnom Penh, Cambodia",11.549942,104.920893,"Agile Sky Residence, ផ្លូវ ៩៥, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា"
15,M Residence,Boeung Keng Kang,"M Residence, Phnom Penh, Cambodia",11.554595,104.925439,"M Residence, ផ្លូវ ២៨២, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា"
19,Urban Village Phase 1,Meanchey,"Urban Village Phase 1, Phnom Penh, Cambodia",11.521907,104.928608,"Urban Village Phase 1, មហាវិថី សម្តេច ហ៊ុន សែន, បឹងទំពុន, សង្កាត់ចាក់អង្រែលើ, ខណ្ឌមានជ័យ, រាជធានីភ្នំពេញ, 120601, ព្រះរាជាណាចក្រ​កម្ពុជា"
22,R&F CITY,Meanchey,"R&F CITY, Phnom Penh, Cambodia",11.526807,104.928286,"R&F City, Morodok Techo Flyover, បឹងទំពុន, សង្កាត់ចាក់អង្រែលើ, ខណ្ឌមានជ័យ, រាជធានីភ្នំពេញ, 120601, ព្រះរាជាណាចក្រ​កម្ពុជា"
24,Embassy Central,Boeung Keng Kang,"Embassy Central, Phnom Penh, Cambodia",11.548895,104.923387,"Embassy Central, ផ្លូវ ៣៥២, សង្កាត់បឹងកេងកងទី ១, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120102, ព្រះរាជាណាចក្រ​កម្ពុជា"
26,Parc 21 Residence,Chamkarmon,"Parc 21 Residence, Phnom Penh, Cambodia",11.541019,104.920394,"Parc 21 Residence, ផ្លូវ ៤៤០, សង្កាត់បឹងត្របែក, ខណ្ឌចំការមន, រាជធានីភ្នំពេញ, 120112, ព្រះរាជាណាចក្រ​កម្ពុជា"
33,The Pinnacle Residence,Chamkarmon,"The Pinnacle Residence, Phnom Penh, Cambodia",11.531753,104.926095,"The Pinnacle Residence, សង្កាត់បឹងត្របែក, សង្កាត់ផ្សារដើមថ្កូវ, ខណ្ឌចំការមន, រាជធានីភ្នំពេញ, 120112, ព្រះរាជាណាចក្រ​កម្ពុជា"


## inside Phnom Penh

In [157]:
retry_found["inside_phnom_penh"] = (
    retry_found.apply(
        lambda row: inside_phnom_penh(
            row["latitude"],
            row["longitude"]
        ),
        axis=1
    )
)

In [158]:
retry_found[
    "inside_phnom_penh"
].value_counts()

inside_phnom_penh
True    15
Name: count, dtype: int64

## Duplicate coordinates

In [159]:
retry_duplicate_coordinates = (
    retry_found[
        retry_found.duplicated(
            subset=[
                "latitude",
                "longitude"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "latitude",
            "longitude"
        ]
    )
)

retry_duplicate_coordinates[
    [
        "project_name",
        "district",
        "latitude",
        "longitude",
        "display_name"
    ]
]

,project_name,district,latitude,longitude,display_name
2,Time Square 306,Boeung Keng Kang,11.555312,104.923949,"Time Square Hotel & Apartment, No 9. E1, ផ្លូវ ២៧៨, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា"
7,Time Square 302,Boeung Keng Kang,11.555312,104.923949,"Time Square Hotel & Apartment, No 9. E1, ផ្លូវ ២៧៨, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា"


## Mark the retry validation

In [161]:
retry_found["coordinate_validation"] = "valid"

In [162]:
# mark the three incorrect

invalid_retry_projects = [
    "Time Square 306",
    "Time Square 302",
    "BK Residence",
]

retry_found.loc[
    retry_found["project_name"].isin(
        invalid_retry_projects
    ),
    "coordinate_validation"
] = "invalid_wrong_match"

In [163]:
# check

retry_found[
    "coordinate_validation"
].value_counts()

coordinate_validation
valid                  12
invalid_wrong_match     3
Name: count, dtype: int64

In [164]:
retry_found[
    [
        "project_name",
        "latitude",
        "longitude",
        "display_name",
        "coordinate_validation"
    ]
]

,project_name,latitude,longitude,display_name,coordinate_validation
2,Time Square 306,11.555312,104.923949,"Time Square Hotel & Apartment, No 9. E1, ផ្លូវ ២៧៨, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា",invalid_wrong_match
7,Time Square 302,11.555312,104.923949,"Time Square Hotel & Apartment, No 9. E1, ផ្លូវ ២៧៨, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា",invalid_wrong_match
10,Vue Aston,11.546288,104.946528,"Vue Aston, Tonle Bassac Promenade, កោះពេជ្រ, សង្កាត់ទន្លេបាសាក់, ខណ្ឌចំការមន, រាជធានីភ្នំពេញ, 120101, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
12,Agile Sky Residence,11.549942,104.920893,"Agile Sky Residence, ផ្លូវ ៩៥, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
15,M Residence,11.554595,104.925439,"M Residence, ផ្លូវ ២៨២, សង្កាត់បឹងកេងកងទី ២, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120103, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
19,Urban Village Phase 1,11.521907,104.928608,"Urban Village Phase 1, មហាវិថី សម្តេច ហ៊ុន សែន, បឹងទំពុន, សង្កាត់ចាក់អង្រែលើ, ខណ្ឌមានជ័យ, រាជធានីភ្នំពេញ, 120601, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
22,R&F CITY,11.526807,104.928286,"R&F City, Morodok Techo Flyover, បឹងទំពុន, សង្កាត់ចាក់អង្រែលើ, ខណ្ឌមានជ័យ, រាជធានីភ្នំពេញ, 120601, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
24,Embassy Central,11.548895,104.923387,"Embassy Central, ផ្លូវ ៣៥២, សង្កាត់បឹងកេងកងទី ១, ខណ្ឌបឹងកេងកង, រាជធានីភ្នំពេញ, 120102, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
26,Parc 21 Residence,11.541019,104.920394,"Parc 21 Residence, ផ្លូវ ៤៤០, សង្កាត់បឹងត្របែក, ខណ្ឌចំការមន, រាជធានីភ្នំពេញ, 120112, ព្រះរាជាណាចក្រ​កម្ពុជា",valid
33,The Pinnacle Residence,11.531753,104.926095,"The Pinnacle Residence, សង្កាត់បឹងត្របែក, សង្កាត់ផ្សារដើមថ្កូវ, ខណ្ឌចំការមន, រាជធានីភ្នំពេញ, 120112, ព្រះរាជាណាចក្រ​កម្ពុជា",valid


## Now combine first-pass + retry coordinates

In [165]:
first_valid = found_projects[
    found_projects[
        "coordinate_validation"
    ] == "valid_first_pass"
].copy()

retry_valid = retry_found[
    retry_found[
        "coordinate_validation"
    ] == "valid"
].copy()

In [166]:
first_valid_coords = first_valid[
    [
        "project_name",
        "district",
        "latitude",
        "longitude",
        "display_name"
    ]
].copy()

first_valid_coords[
    "geocode_method"
] = "first_pass"

In [167]:
retry_valid_coords = retry_valid[
    [
        "project_name",
        "district",
        "latitude",
        "longitude",
        "display_name"
    ]
].copy()

retry_valid_coords[
    "geocode_method"
] = "retry_simple_query"

In [168]:
validated_project_coords = pd.concat(
    [
        first_valid_coords,
        retry_valid_coords
    ],
    ignore_index=True
)

In [174]:
print(
    "Validated project coordinates:",
    len(validated_project_coords)
)

Validated project coordinates: 27


## One final duplicate-coordinate check

In [172]:
final_coordinate_duplicates = (
    validated_project_coords[
        validated_project_coords.duplicated(
            subset=[
                "latitude",
                "longitude"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "latitude",
            "longitude"
        ]
    )
)

final_coordinate_duplicates

,project_name,district,latitude,longitude,display_name,geocode_method


## Save the validated coordinate table

In [173]:
validated_project_coords.to_csv(
    "../data/geo/"
    "validated_project_coordinates.csv",
    index=False,
    encoding="utf-8-sig"
)

So none of the validated projects share an identical coordinate with another project. The validated project coordinates are now ready to map back to your property listings. 📍

## map coordinates back to the 2,673 listings

In [175]:
print(
    "Validated projects:",
    len(validated_project_coords)
)

print(
    "Unique project names:",
    validated_project_coords[
        "project_name"
    ].nunique()
)

Validated projects: 27
Unique project names: 27


## Create coordinate mappings

In [176]:
lat_map = (
    validated_project_coords
    .set_index("project_name")["latitude"]
    .to_dict()
)

lon_map = (
    validated_project_coords
    .set_index("project_name")["longitude"]
    .to_dict()
)

method_map = (
    validated_project_coords
    .set_index("project_name")["geocode_method"]
    .to_dict()
)

## Find listings belonging to those projects

In [177]:
project_coord_mask = (
    location_df["project_name"]
    .isin(
        validated_project_coords[
            "project_name"
        ]
    )
)

print(
    "Listings with validated project coordinates:",
    project_coord_mask.sum()
)

print(
    "Projects represented:",
    location_df.loc[
        project_coord_mask,
        "project_name"
    ].nunique()
)

Listings with validated project coordinates: 141
Projects represented: 27


## Fill latitude and longitude

In [178]:
location_df.loc[
    project_coord_mask,
    "latitude"
] = (
    location_df.loc[
        project_coord_mask,
        "project_name"
    ]
    .map(lat_map)
)

location_df.loc[
    project_coord_mask,
    "longitude"
] = (
    location_df.loc[
        project_coord_mask,
        "project_name"
    ]
    .map(lon_map)
)

## 4. Add coordinate quality tracking

This is important.

In [179]:
location_df[
    "coordinate_source"
] = None

location_df[
    "coordinate_precision"
] = None

In [180]:
location_df.loc[
    project_coord_mask,
    "coordinate_source"
] = (
    location_df.loc[
        project_coord_mask,
        "project_name"
    ]
    .map(method_map)
)

In [181]:
location_df.loc[
    project_coord_mask,
    "coordinate_precision"
] = "project_building"

## Check coordinate coverage

In [182]:
coordinate_complete = (
    location_df["latitude"].notna()
    &
    location_df["longitude"].notna()
)

print(
    "Total listings:",
    len(location_df)
)

print(
    "Listings with coordinates:",
    coordinate_complete.sum()
)

print(
    "Coordinate coverage:",
    f"{coordinate_complete.mean() * 100:.1f}%"
)

Total listings: 2673
Listings with coordinates: 141
Coordinate coverage: 5.3%


In [183]:
location_df[
    "coordinate_precision"
].value_counts(
    dropna=False
)

coordinate_precision
None                2532
project_building     141
Name: count, dtype: int64

In [ ]:
# inspect complete coordinates

location_df[
    coordinate_complete
][
    [
        "listing_id",
        "project_name",
        "district",
        "latitude",
        "longitude",
        "coordinate_source",
        "coordinate_precision"
    ]
].head(30)

,listing_id,project_name,district,latitude,longitude,coordinate_source,coordinate_precision
3,realestate.com.kh_255697,One Park,Daun Penh,11.576143,104.905704,first_pass,project_building
15,realestate.com.kh_267254,Time Square 3,Toul Kork,11.583571,104.897168,first_pass,project_building
23,realestate.com.kh_267198,North Park Condominium,Sen Sok,11.551292,104.872536,first_pass,project_building
24,realestate.com.kh_267184,Yuetai Phnom Penh Harbour,Daun Penh,11.581475,104.922626,first_pass,project_building
25,realestate.com.kh_267171,Vue Aston,Chbar Ampov,11.546288,104.946528,retry_simple_query,project_building
29,realestate.com.kh_267075,Agile Sky Residence,Boeung Keng Kang,11.549942,104.920893,retry_simple_query,project_building
31,realestate.com.kh_267071,Arakawa Residence,Sen Sok,11.559708,104.880765,first_pass,project_building
34,realestate.com.kh_267023,Vue Aston,Chbar Ampov,11.546288,104.946528,retry_simple_query,project_building
37,realestate.com.kh_266959,Vue Aston,Chbar Ampov,11.546288,104.946528,retry_simple_query,project_building
42,realestate.com.kh_266915,M Residence,Boeung Keng Kang,11.554595,104.925439,retry_simple_query,project_building


## Very important validation

In [185]:
print(location_df.shape)

(2673, 33)


In [187]:
coordinate_complete = (
    location_df["latitude"].notna()
    &
    location_df["longitude"].notna()
)

print(
    "Listings with coordinates:",
    coordinate_complete.sum()
)

print(
    "Coordinate coverage:",
    f"{coordinate_complete.mean() * 100:.1f}%"
)

print(
    "Missing coordinates:",
    (~coordinate_complete).sum()
)

print(
    "Total rows:",
    len(location_df)
)

Listings with coordinates: 141
Coordinate coverage: 5.3%
Missing coordinates: 2532
Total rows: 2673


## Save a new Gold file

In [188]:
location_df.to_csv(
    "../data/gold/"
    "property_listings_geocoded.csv",
    index=False,
    encoding="utf-8-sig"
)

## Location Recovery Summary

- Total listings remained unchanged at 2,673.
- District coverage improved from 84.6% to approximately 91.8%.
- Missing districts were recovered only when reliable textual or commune evidence was available.
- 27 condominium projects were successfully validated and geocoded.
- These project coordinates were mapped to 141 listings, giving approximately 5.3% reliable coordinate coverage.
- Approximate district-level coordinates were not assigned because they would not represent exact property locations.
- District will therefore be the primary location feature in the main prediction model.
- Coordinate and distance features may be evaluated separately on the subset with reliable project-level locations.